# Figure 2. Model Performance Metrics

loads per-individual JSON metrics from `ml_out_var/metrics/test_*.json`.
models: `splaire_ref`, `splaire_var`, `spliceai`, `pangolin`, `splicetransformer`.

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
from pathlib import Path

ANNOT_SIZE = 12
plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 14, "figure.titlesize": 18,
    "axes.titlesize": 16, "axes.labelsize": 14, "axes.titleweight": "normal",
    "xtick.labelsize": 12, "ytick.labelsize": 12,
    "legend.fontsize": 12, "legend.title_fontsize": 14, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42,
})

GRAY_DARK, GRAY_MED, GRAY_LIGHT = "#404040", "#666666", "#888888"
GRAY_FAINT, GRAY_BG, GRAY_LINE = "#cccccc", "#e0e0e0", "#5a5a5a"

model_colors = {
    "pangolin": "#009E73", "spliceai": "#D55E00", "splicetransformer": "#E69F00",
    "splaire_ref": "#56B4E9", "splaire_var": "#0072B2", "splaire_avg": "#CC79A7",
    "gencode": "#666666",
}
tissue_colors = {
    "haec": "#7570b3", "haec10": "#7570b3", "lung": "#1b9e77", "testis": "#d95f02",
    "brain_cortex": "#e7298a", "whole_blood": "#66a61e",
}
tissue_display = {
    "haec": "HAEC", "haec10": "HAEC", "lung": "Lung", "testis": "Testis",
    "brain_cortex": "Brain", "whole_blood": "Blood", "gencode": "GENCODE", "mane_select": "MANE",
}
model_display = {
    "pangolin": "Pangolin", "spliceai": "SpliceAI", "splicetransformer": "SpliceTransformer",
    "splaire_ref": "SPLAIRE", "splaire_var": "SPLAIRE (var)", "splaire_avg": "SPLAIRE (avg)",
}
r2_outputs = ["splaire_ref_ssu", "spliceai_cls", "pangolin_avg_usage", "splicetransformer_avg_tissue"]

def _get_base_model(key):
    key = key.lower().replace("-", "_").replace(" ", "_")
    if key in model_colors: return key
    for b in sorted(model_colors, key=len, reverse=True):
        if key.startswith(b): return b
    return None

def get_color(key):
    b = _get_base_model(key)
    return model_colors[b] if b else "#999999"

def get_name(key):
    b = _get_base_model(key)
    return model_display.get(b, key)

def _match_tissue(desired, available):
    for t in available:
        if desired in t.lower(): return t
    return None

tissues_fig1 = ["haec10", "lung", "testis", "brain_cortex", "whole_blood"]
tissues = list(tissues_fig1)
base = Path("/scratch/runyan.m/sphaec_out")
canonical_base = Path("/scratch/runyan.m/splaire_out/canonical")

# canonical benchmarks (classification only, no per-individual variation)
gencode_tissues = ["gencode", "mane_select"]

# pangolin test set (4 tissues, each scored separately)
pang_test_tissues = ["heart", "liver", "brain", "testis"]
pang_tissue_display = {"heart": "Heart", "liver": "Liver", "brain": "Brain", "testis": "Testis"}

fig2_main = Path("figures/fig2/main")
fig2_sup = Path("figures/fig2/sup")
fig2_main.mkdir(parents=True, exist_ok=True)
fig2_sup.mkdir(parents=True, exist_ok=True)

exclude_bases = {"splaire_var", "splaire_avg"}
model_order = ["splaire_ref", "spliceai", "pangolin", "splicetransformer"]
metrics_subdir = "ml_out_var/metrics"

print("config loaded")

In [ ]:
# load all metrics from JSON files
cls_rows, reg_rows, bin_cls_ratio_rows, bin_reg_rows = [], [], [], []
n_files_loaded = 0

cls_subsets = ["all", "ssu_valid", "ssu_valid_nonzero", "ssu_shared", "ssu_shared_nonzero"]
reg_subsets_list = ["ssu_valid", "ssu_valid_nonzero", "ssu_shared", "ssu_shared_nonzero"]


def _load_json_metrics(jp, tissue, sid):
    """parse one metrics JSON into cls/reg/binned row lists"""
    with open(jp) as fp:
        d = json.load(fp)["overall"]

    # classification
    for subset in cls_subsets:
        for target in ["acceptor", "donor"]:
            grp = d.get("classification", {}).get(subset, {}).get(target, {})
            for output, m in grp.items():
                row = {"tissue": tissue, "sample_id": sid, "subset": subset,
                       "target": target, "output": output,
                       "auprc": float(m["auprc"]), "auroc": float(m["auroc"]),
                       "topk": float(m["topk"]),
                       "n_pos": int(m["n_pos"]), "n_neg": int(m.get("n_neg", 0))}
                if "f1_max" in m:
                    row["f1_max"] = float(m["f1_max"])
                cls_rows.append(row)

    # regression
    for subset in reg_subsets_list:
        grp = d.get("regression", {}).get(subset, {})
        for output, m in grp.items():
            p = m.get("pearson")
            if p is None or (isinstance(p, float) and p != p):
                continue  # skip NaN
            row = {"tissue": tissue, "sample_id": sid, "subset": subset,
                   "output": output,
                   "pearson": float(p), "spearman": float(m["spearman"]),
                   "r2": float(m["r2"]), "mse": float(m["mse"]), "n": int(m["n"])}
            if "mae" in m: row["mae"] = float(m["mae"])
            reg_rows.append(row)

    # binned classification (ratio-preserving)
    for subset in ["ssu_valid", "ssu_valid_nonzero"]:
        for target in ["acceptor", "donor"]:
            grp = d.get("binned", {}).get("classification_ratio", {}).get(subset, {}).get(target, {})
            for output, bins in grp.items():
                for bk, m in bins.items():
                    bin_cls_ratio_rows.append({
                        "tissue": tissue, "sample_id": sid, "subset": subset,
                        "target": target, "output": output, "bin": int(bk.replace("bin_", "")),
                        "auprc": float(m["auprc"]), "auroc": float(m["auroc"]),
                        "topk": float(m["topk"]),
                        "n_pos": int(m["n_pos"]), "n_neg": int(m.get("n_neg", 0))})

    # binned regression
    for subset in reg_subsets_list:
        grp = d.get("binned", {}).get("regression", {}).get(subset, {})
        for output, bins in grp.items():
            for bk, m in bins.items():
                p = m.get("pearson")
                if p is None or (isinstance(p, float) and p != p):
                    continue
                row = {"tissue": tissue, "sample_id": sid, "subset": subset,
                       "output": output, "bin": int(bk.replace("bin_", "")),
                       "pearson": float(p), "spearman": float(m["spearman"]),
                       "r2": float(m["r2"]), "mse": float(m["mse"]), "n": int(m["n"])}
                if "mae" in m: row["mae"] = float(m["mae"])
                bin_reg_rows.append(row)


# --- expression tissues (per-individual JSON files) ---
for tissue in tissues:
    json_dir = base / tissue / metrics_subdir
    if not json_dir.exists():
        print(f"{tissue}: not found")
        continue
    json_files = sorted(json_dir.glob("test_*.json"))
    print(f"{tissue}: {len(json_files)} files")

    for jp in json_files:
        sid = jp.stem.replace("test_", "")
        n_files_loaded += 1
        _load_json_metrics(jp, tissue, sid)

# --- canonical benchmarks (gencode, mane_select) ---
canonical_metrics = {
    "gencode": canonical_base / "gencode" / "gencode_metrics.json",
    "mane_select": canonical_base / "mane_select" / "mane_select_metrics.json",
}
for tissue, jp in canonical_metrics.items():
    if not jp.exists():
        print(f"{tissue}: not found at {jp}")
        continue
    n_files_loaded += 1
    _load_json_metrics(jp, tissue, "reference")
    print(f"{tissue}: {jp.name}")

# --- pangolin test set (per-tissue JSON files) ---
pang_cls_rows, pang_reg_rows = [], []
pang_test_dir = canonical_base / "pangolin"
for pt in pang_test_tissues:
    jp = pang_test_dir / f"{pt}_metrics.json"
    if not jp.exists():
        print(f"pangolin {pt}: not found at {jp}")
        continue
    with open(jp) as fp:
        d = json.load(fp)["overall"]
    print(f"pangolin {pt}: {jp.name}")

    # classification (all subset)
    for target in ["acceptor", "donor"]:
        grp = d.get("classification", {}).get("all", {}).get(target, {})
        for output, m in grp.items():
            pang_cls_rows.append({
                "tissue": pt, "output": output,
                "auprc": float(m["auprc"]), "auroc": float(m["auroc"]),
                "topk": float(m["topk"]),
                "n_pos": int(m["n_pos"]), "n_neg": int(m.get("n_neg", 0)),
                "f1": float(m.get("f1_max", m["topk"])),
            })

    # regression
    for subset in ["ssu_valid", "ssu_valid_nonzero"]:
        grp = d.get("regression", {}).get(subset, {})
        for output, m in grp.items():
            p = m.get("pearson")
            if p is None or (isinstance(p, float) and p != p):
                continue
            pang_reg_rows.append({
                "tissue": pt, "subset": subset, "output": output,
                "pearson": float(p), "spearman": float(m["spearman"]),
                "r2": float(m["r2"]), "mse": float(m["mse"]), "n": int(m["n"]),
            })

df_pang_cls = pd.DataFrame(pang_cls_rows)
df_pang_reg = pd.DataFrame(pang_reg_rows)

# --- build dataframes ---
df_cls = pd.DataFrame(cls_rows)
df_reg = pd.DataFrame(reg_rows)
df_bin_ratio = pd.DataFrame(bin_cls_ratio_rows) if bin_cls_ratio_rows else pd.DataFrame(
    columns=["tissue","sample_id","subset","target","output","bin","auprc","auroc","topk","n_pos","n_neg"])
df_bin_reg = pd.DataFrame(bin_reg_rows) if bin_reg_rows else pd.DataFrame(
    columns=["tissue","sample_id","subset","output","bin","mse","n"])
all_tissues = tissues + [t for t in gencode_tissues if t in df_cls.tissue.unique()]

print(f"\nloaded {n_files_loaded} files")
print(f"df_cls: {df_cls.shape}, df_reg: {df_reg.shape}")
print(f"df_bin_ratio: {df_bin_ratio.shape}, df_bin_reg: {df_bin_reg.shape}")
print(f"all_tissues: {all_tissues}")
if len(df_cls) > 0: print(f"cls outputs: {sorted(df_cls.output.unique())}")
if len(df_reg) > 0: print(f"reg outputs: {sorted(df_reg.output.unique())}")
if len(df_pang_cls) > 0: print(f"pangolin cls: {df_pang_cls.shape}, tissues: {sorted(df_pang_cls.tissue.unique())}")
if len(df_pang_reg) > 0: print(f"pangolin reg: {df_pang_reg.shape}, tissues: {sorted(df_pang_reg.tissue.unique())}")

## Supplemental: Heatmaps
red solid = best overall, orange dashed = best per model family

In [ ]:
lower_better = {"mse", "mae"}
_output_display = {
    "splaire_ref": "SPLAIRE CLS", "splaire_ref_ssu": "SPLAIRE SSU",
    "spliceai": "SpliceAI CLS", "splicetransformer": "SpliceTransformer CLS",
    "splicetransformer_avg_tissue": "SpliceTransformer Avg Tissue",
}

def _output_name(code):
    if code in _output_display: return _output_display[code]
    if code.startswith("pangolin_"):
        rest = code.replace("pangolin_", "")
        if rest.endswith("_p_splice"):
            return f"Pangolin {rest.replace('_p_splice','').replace('_',' ').title()} CLS"
        if rest.endswith("_usage"):
            return f"Pangolin {rest.replace('_usage','').replace('_',' ').title()} Usage"
    if code.startswith("splicetransformer_"):
        return f"SpliceTransformer {code.replace('splicetransformer_','').replace('_',' ').title()}"
    return code

def plot_heatmap(data, metric, title_suffix="", fname_suffix="", out_dir=None, tissue_list=None):
    if tissue_list is None: tissue_list = all_tissues
    sub = data[data.metric == metric]
    if len(sub) == 0: return
    pivot = sub.pivot(index="output", columns="tissue", values="score")
    pivot = pivot[[t for t in tissue_list if t in pivot.columns]]
    if pivot.empty: return
    ascending = metric in lower_better
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=ascending).index]
    fig, ax = plt.subplots(figsize=(10, max(4, len(pivot)*0.4)))
    im = ax.imshow(pivot.values, cmap="YlGnBu", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([tissue_display.get(t,t) for t in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([_output_name(o) for o in pivot.index])
    n_expr = len([t for t in pivot.columns if t in set(tissues_fig1)])
    if n_expr < len(pivot.columns):
        ax.axvline(x=n_expr - 0.5, color="white", lw=2, ls="--")
    o2b = {o: _get_base_model(o) for o in pivot.index}
    bms = set(o2b.values()) - {None}
    for j in range(len(pivot.columns)):
        cv = pivot.iloc[:,j]
        for bm in bms:
            bo = [o for o,b in o2b.items() if b==bm]
            bv = cv.loc[bo]
            best = bv.idxmin() if metric in lower_better else bv.idxmax()
            i = list(pivot.index).index(best)
            ax.add_patch(Rectangle((j-0.5,i-0.5),1,1,fill=False,edgecolor="orange",lw=1.5,ls="--"))
        bi = cv.idxmin() if metric in lower_better else cv.idxmax()
        i = list(pivot.index).index(bi)
        ax.add_patch(Rectangle((j-0.5,i-0.5),1,1,fill=False,edgecolor="red",lw=2.5))
    vmin,vmax = pivot.values.min(), pivot.values.max()
    mid = (vmin+vmax)/2
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.iloc[i,j]
            ax.text(j,i,f"{v:.2f}",ha="center",va="center",color="white" if v>mid else "black",fontsize=ANNOT_SIZE)
    plt.colorbar(im,ax=ax,shrink=0.5)
    ax.set_title(f"{metric.upper()}{title_suffix}")
    ax.legend(handles=[Patch(facecolor="none",edgecolor="red",lw=2.5,label="best overall"),
                       Patch(facecolor="none",edgecolor="orange",lw=1.5,ls="--",label="best per model")],
              loc="upper left",bbox_to_anchor=(1.15,1))
    plt.tight_layout()
    if out_dir: fig.savefig(out_dir / f"heatmap_{metric}{fname_suffix}.png", dpi=300)
    plt.show()

# classification heatmaps (includes gencode/mane if loaded)
_heatmap_tissues_cls = tissues_fig1 + [t for t in gencode_tissues if t in df_cls.tissue.unique()]
if len(df_cls) > 0:
    sub = df_cls.query("subset == 'all'").copy()
    sub["base_model"] = sub.output.apply(_get_base_model)
    sub = sub[~sub.base_model.isin(exclude_bases)]
    agg = sub.groupby(["tissue","output"]).agg(auprc=("auprc","mean"),auroc=("auroc","mean"),topk=("topk","mean")).reset_index()
    cl = agg.melt(id_vars=["tissue","output"],var_name="metric",value_name="score")
    for m in ["auprc","auroc","topk"]:
        plot_heatmap(cl, m, tissue_list=_heatmap_tissues_cls, out_dir=fig2_sup)

# regression heatmaps (expression tissues only — canonical has no regression)
subset_labels = {"ssu_valid":"all sites","ssu_valid_nonzero":"SSU > 0",
                 "ssu_shared":"shared sites","ssu_shared_nonzero":"shared, SSU > 0"}
if len(df_reg) > 0:
    for subset, slabel in subset_labels.items():
        sub = df_reg.query("subset == @subset").copy()
        sub["base_model"] = sub.output.apply(_get_base_model)
        sub = sub[~sub.base_model.isin(exclude_bases)]
        if len(sub) == 0: continue
        ac = ["pearson","spearman","r2","mse"]
        if "mae" in sub.columns: ac.append("mae")
        agg = sub.groupby(["tissue","output"]).agg({c:"mean" for c in ac}).reset_index()
        rl = agg.melt(id_vars=["tissue","output"],var_name="metric",value_name="score")
        for m in ac:
            plot_heatmap(rl, m, title_suffix=f" ({slabel})", fname_suffix=f"_{subset}",
                         tissue_list=tissues_fig1, out_dir=fig2_sup)

# Figure 2a: AUPRC stratified by SSU

In [ ]:
bin_labels = [f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)]
cls_order = ["splaire_ref", "pangolin_avg_p_splice", "splicetransformer", "spliceai"]
reg_order = ["splaire_ref_ssu", "pangolin_avg_usage", "splicetransformer_avg_tissue"]
all_order = cls_order + reg_order

_desired_v = ["haec", "lung", "brain", "testis", "blood"]
tissues_v_order = [m for d in _desired_v if (m := _match_tissue(d, tissues_fig1)) is not None]

# per-tissue individual plots
for tissue in tissues:
    sub = df_bin_ratio.query("tissue == @tissue & subset == 'ssu_valid_nonzero'").copy()
    if len(sub) == 0: continue
    avail = [o for o in all_order if o in sub.output.unique()]
    if not avail: continue
    sub = sub[sub.output.isin(avail)]
    agg = sub.groupby(["output","bin"]).agg(auprc=("auprc","mean"),n_pos=("n_pos","mean")).reset_index()
    counts = agg[agg.output==avail[0]][["bin","n_pos"]].sort_values("bin")

    fig, ax = plt.subplots(figsize=(6,6))
    ax2 = ax.twinx()
    ax2.bar(counts["bin"],counts["n_pos"],width=0.5,color=GRAY_FAINT,alpha=0.4,zorder=1)
    ax2.set_ylabel("n sites",color=GRAY_MED); ax2.tick_params(axis="y",labelcolor=GRAY_MED)
    ax2.set_yscale("log"); ax2.set_ylim(counts["n_pos"].min()*0.5,counts["n_pos"].max()*3)
    for out in avail:
        od = agg[agg.output==out].sort_values("bin")
        ax.plot(od["bin"],od["auprc"],marker="o",label=get_name(out),color=get_color(out),
                lw=2,markersize=6,zorder=3,ls="--" if out in reg_order else "-")
    ax.set_xticks(range(10)); ax.set_xticklabels(bin_labels,rotation=45,ha="right")
    ax.set_xlabel("SSU bin"); ax.set_ylabel("AUPRC"); ax.set_ylim(0,1)
    ax.legend(loc="upper center",bbox_to_anchor=(0.5,1.15),ncol=4,frameon=False)
    ax.set_title(tissue_display.get(tissue,tissue))
    ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
    plt.tight_layout(); plt.savefig(fig2_main / f"binned_auprc_{tissue}.png",dpi=300); plt.show()

# vertical stacked
nt_v = len(tissues_v_order)
fig, axes = plt.subplots(nt_v,1,figsize=(6,4*nt_v),sharex=True)
for row, tissue in enumerate(tissues_v_order):
    ax = axes[row]
    sub = df_bin_ratio.query("tissue == @tissue & subset == 'ssu_valid_nonzero'").copy()
    if len(sub) == 0:
        ax.text(0.5,0.5,"no data",ha="center",va="center",transform=ax.transAxes)
        ax.set_title(tissue_display.get(tissue,tissue)); continue
    avail = [o for o in all_order if o in sub.output.unique()]
    sub = sub[sub.output.isin(avail)]
    agg = sub.groupby(["output","bin"]).agg(auprc=("auprc","mean"),n_pos=("n_pos","mean")).reset_index()
    counts = agg[agg.output==avail[0]][["bin","n_pos"]].sort_values("bin")
    ax2 = ax.twinx()
    ax2.bar(counts["bin"],counts["n_pos"],width=0.5,color=GRAY_FAINT,alpha=0.4,zorder=1)
    ax2.set_ylabel("n sites" if row==nt_v//2 else "",color=GRAY_MED)
    ax2.tick_params(axis="y",labelcolor=GRAY_MED); ax2.set_yscale("log")
    ax2.set_ylim(counts["n_pos"].min()*0.5,counts["n_pos"].max()*3)
    for out in avail:
        od = agg[agg.output==out].sort_values("bin")
        ax.plot(od["bin"],od["auprc"],marker="o",color=get_color(out),lw=2,markersize=6,zorder=3,
                ls="--" if out in reg_order else "-")
    ax.set_ylim(0,1); ax.set_ylabel("AUPRC")
    ax.set_title(tissue_display.get(tissue,tissue))
    ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
    if row == nt_v-1:
        ax.set_xticks(range(10)); ax.set_xticklabels(bin_labels,rotation=45,ha="right")
        ax.set_xlabel("SSU bin")
    else:
        ax.set_xticklabels([]); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(fig2_main / "binned_auprc_v.png",dpi=300,bbox_inches="tight"); plt.show()

# horizontal stacked
nt_h = len(tissues_v_order)
fig, axes = plt.subplots(1,nt_h,figsize=(6*nt_h,5),sharey=True)
for col, tissue in enumerate(tissues_v_order):
    ax = axes[col]
    sub = df_bin_ratio.query("tissue == @tissue & subset == 'ssu_valid_nonzero'").copy()
    if len(sub) == 0:
        ax.set_title(tissue_display.get(tissue,tissue)); continue
    avail = [o for o in all_order if o in sub.output.unique()]
    sub = sub[sub.output.isin(avail)]
    agg = sub.groupby(["output","bin"]).agg(auprc=("auprc","mean"),n_pos=("n_pos","mean")).reset_index()
    counts = agg[agg.output==avail[0]][["bin","n_pos"]].sort_values("bin")
    ax2 = ax.twinx()
    ax2.bar(counts["bin"],counts["n_pos"],width=0.5,color=GRAY_FAINT,alpha=0.4,zorder=1)
    ax2.tick_params(axis="y",labelcolor=GRAY_MED); ax2.set_yscale("log")
    ax2.set_ylim(counts["n_pos"].min()*0.5,counts["n_pos"].max()*3)
    if col==nt_h-1: ax2.set_ylabel("n sites",color=GRAY_MED)
    else: ax2.set_yticklabels([])
    for out in avail:
        od = agg[agg.output==out].sort_values("bin")
        ax.plot(od["bin"],od["auprc"],marker="o",color=get_color(out),lw=2,markersize=6,zorder=3,
                ls="--" if out in reg_order else "-")
    ax.set_ylim(0,1)
    if col==0: ax.set_ylabel("AUPRC")
    ax.set_title(tissue_display.get(tissue,tissue))
    ax.set_xticks(range(10)); ax.set_xticklabels(bin_labels,rotation=45,ha="right"); ax.set_xlabel("SSU bin")
    ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
plt.tight_layout()
plt.savefig(fig2_main / "binned_auprc_h.png",dpi=300,bbox_inches="tight"); plt.show()

# Figure 2b: MSE stratified by SSU

In [ ]:
output_order_spec = [
    "splaire_ref_cls","splaire_ref_ssu","pangolin_avg_p_splice","pangolin_avg_usage",
    "splicetransformer_cls","splicetransformer_avg_tissue","spliceai_cls",
]
reg_outputs = ["splaire_ref_ssu","pangolin_avg_usage","splicetransformer_avg_tissue"]

# per-tissue individual plots
for tissue in tissues:
    sub = df_bin_reg.query("tissue == @tissue & subset == 'ssu_valid_nonzero'").copy()
    if len(sub) == 0: continue
    avail = [o for o in output_order_spec if o in sub.output.unique()]
    if not avail: continue
    sub = sub[sub.output.isin(avail)]
    mc = "mae" if "mae" in sub.columns else "mse"
    agg = sub.groupby(["output","bin"]).agg(metric=(mc,"mean"),n=("n","mean")).reset_index()
    counts = agg[agg.output==avail[0]][["bin","n"]].sort_values("bin")

    fig, ax = plt.subplots(figsize=(8,8))
    ax2 = ax.twinx()
    ax2.bar(counts["bin"],counts["n"],width=0.5,color=GRAY_FAINT,alpha=0.4,zorder=1)
    ax2.set_ylabel("n sites",color=GRAY_MED); ax2.tick_params(axis="y",labelcolor=GRAY_MED)
    ax2.set_yscale("log"); ax2.set_ylim(counts["n"].min()*0.5,counts["n"].max()*3)
    for out in avail:
        od = agg[agg.output==out].sort_values("bin")
        is_reg = out in reg_outputs
        ax.plot(od["bin"],od["metric"],marker="o",label=get_name(out)+(" - SSU" if is_reg else ""),
                color=get_color(out),lw=2,markersize=6,zorder=3,ls="--" if is_reg else "-")
    ax.set_xticks(range(10)); ax.set_xticklabels(bin_labels,rotation=45,ha="right")
    ax.set_xlabel("SSU bin"); ax.set_ylabel("Mean Squared Error")
    ax.legend(loc="upper center",bbox_to_anchor=(0.5,1.28),ncol=2,frameon=False,handlelength=3)
    ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
    ax.set_title(tissue_display.get(tissue,tissue))
    plt.tight_layout(); plt.savefig(fig2_main / f"binned_regression_{tissue}.png",dpi=300); plt.show()

# vertical stacked
_mc = "mae" if "mae" in df_bin_reg.columns else "mse"
_y_vals = []
for t in tissues_v_order:
    sv = df_bin_reg.query("tissue == @t & subset == 'ssu_valid_nonzero'")
    if len(sv) > 0:
        av = [o for o in output_order_spec if o in sv.output.unique()]
        sv = sv[sv.output.isin(av)]
        _y_vals.extend(sv.groupby(["output","bin"]).agg(metric=(_mc,"mean")).reset_index()["metric"].tolist())
_y_max_v = max(_y_vals) * 1.1 if _y_vals else 1.0

nt_v = len(tissues_v_order)
fig, axes = plt.subplots(nt_v,1,figsize=(8,4*nt_v),sharex=True)
for row, tissue in enumerate(tissues_v_order):
    ax = axes[row]
    sub = df_bin_reg.query("tissue == @tissue & subset == 'ssu_valid_nonzero'").copy()
    if len(sub) == 0:
        ax.text(0.5,0.5,"no data",ha="center",va="center",transform=ax.transAxes)
        ax.set_title(tissue_display.get(tissue,tissue)); continue
    avail = [o for o in output_order_spec if o in sub.output.unique()]
    sub = sub[sub.output.isin(avail)]
    agg = sub.groupby(["output","bin"]).agg(metric=(_mc,"mean"),n=("n","mean")).reset_index()
    counts = agg[agg.output==avail[0]][["bin","n"]].sort_values("bin")
    ax2 = ax.twinx()
    ax2.bar(counts["bin"],counts["n"],width=0.5,color=GRAY_FAINT,alpha=0.4,zorder=1)
    ax2.set_ylabel("n sites" if row==nt_v//2 else "",color=GRAY_MED)
    ax2.tick_params(axis="y",labelcolor=GRAY_MED); ax2.set_yscale("log")
    ax2.set_ylim(counts["n"].min()*0.5,counts["n"].max()*3)
    for out in avail:
        od = agg[agg.output==out].sort_values("bin")
        ax.plot(od["bin"],od["metric"],marker="o",color=get_color(out),lw=2,markersize=6,zorder=3,
                ls="--" if out in reg_outputs else "-")
    ax.set_ylabel("Mean Squared Error")
    ax.set_title(tissue_display.get(tissue,tissue))
    ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
    if row == nt_v-1:
        ax.set_xticks(range(10)); ax.set_xticklabels(bin_labels,rotation=45,ha="right")
        ax.set_xlabel("SSU bin")
    else:
        ax.set_xticklabels([]); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(fig2_main / "binned_regression_v.png",dpi=300,bbox_inches="tight"); plt.show()

# horizontal stacked
nt_h = len(tissues_v_order)
fig, axes = plt.subplots(1,nt_h,figsize=(6*nt_h,5),sharey=True)
for col, tissue in enumerate(tissues_v_order):
    ax = axes[col]
    sub = df_bin_reg.query("tissue == @tissue & subset == 'ssu_valid_nonzero'").copy()
    if len(sub) == 0:
        ax.set_title(tissue_display.get(tissue,tissue)); continue
    avail = [o for o in output_order_spec if o in sub.output.unique()]
    sub = sub[sub.output.isin(avail)]
    agg = sub.groupby(["output","bin"]).agg(metric=(_mc,"mean"),n=("n","mean")).reset_index()
    counts = agg[agg.output==avail[0]][["bin","n"]].sort_values("bin")
    ax2 = ax.twinx()
    ax2.bar(counts["bin"],counts["n"],width=0.5,color=GRAY_FAINT,alpha=0.4,zorder=1)
    ax2.tick_params(axis="y",labelcolor=GRAY_MED); ax2.set_yscale("log")
    ax2.set_ylim(counts["n"].min()*0.5,counts["n"].max()*3)
    if col==nt_h-1: ax2.set_ylabel("n sites",color=GRAY_MED)
    else: ax2.set_yticklabels([])
    for out in avail:
        od = agg[agg.output==out].sort_values("bin")
        ax.plot(od["bin"],od["metric"],marker="o",color=get_color(out),lw=2,markersize=6,zorder=3,
                ls="--" if out in reg_outputs else "-")
    if col==0: ax.set_ylabel("Mean Squared Error")
    ax.set_title(tissue_display.get(tissue,tissue))
    ax.set_xticks(range(10)); ax.set_xticklabels(bin_labels,rotation=45,ha="right"); ax.set_xlabel("SSU bin")
    ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
plt.tight_layout()
plt.savefig(fig2_main / "binned_regression_h.png",dpi=300,bbox_inches="tight"); plt.show()

## Supplemental: Sample Sizes
strip plots and summary tables for classification (n positives) and regression (n splice sites) by tissue and subset.

In [ ]:
_cls_labels = {"all": "all positions", **subset_labels}

# classification: n_pos by tissue, per subset
if len(df_cls) > 0:
    cls_subsets_plot = [s for s in ["all","ssu_valid","ssu_valid_nonzero","ssu_shared","ssu_shared_nonzero"]
                        if s in df_cls.subset.unique()]
    for subset in cls_subsets_plot:
        n_df = df_cls[df_cls.subset == subset].groupby(["tissue","sample_id","target"]).agg(
            n_pos=("n_pos","first"), n_neg=("n_neg","first")).reset_index()
        cls_tissues = [t for t in all_tissues if t in n_df.tissue.unique()]
        n_df = n_df[n_df.tissue.isin(cls_tissues)]
        if len(n_df) == 0: continue

        fig, axes = plt.subplots(1,2,figsize=(12,5))
        for ax, target in zip(axes, ["acceptor","donor"]):
            tsub = n_df[n_df.target == target]
            sns.stripplot(data=tsub, x="tissue", y="n_pos", hue="tissue", order=cls_tissues,
                          palette={t: tissue_colors.get(t,"#999999") for t in cls_tissues},
                          size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)
            for i, tissue in enumerate(cls_tissues):
                vals = tsub[tsub.tissue == tissue].n_pos
                if len(vals) > 0:
                    ax.hlines(vals.mean(), i-0.3, i+0.3, color=tissue_colors.get(tissue,"#999999"), lw=2.5)
            ax.set_xlabel(""); ax.set_ylabel("n positives")
            ax.set_xticks(range(len(cls_tissues)))
            ax.set_xticklabels([tissue_display.get(t,t) for t in cls_tissues], rotation=45, ha="right")
            ax.set_title(f"{target.capitalize()} sites")

        fig.suptitle(f"Classification \u2014 {_cls_labels.get(subset, subset)}", y=1.02)
        plt.tight_layout()
        plt.savefig(fig2_sup / f"sample_sizes_cls_{subset}.png", dpi=300, bbox_inches="tight")
        plt.show()

        # summary table
        summary = n_df.groupby(["tissue","target"]).agg(
            n_pos_mean=("n_pos","mean"), n_pos_std=("n_pos","std"),
            n_neg_mean=("n_neg","mean"), n_samples=("sample_id","nunique")).reset_index()
        def fmt_n(row):
            if pd.isna(row.n_pos_std) or row.n_pos_std == 0:
                return f"{row.n_pos_mean:,.0f}"
            return f"{row.n_pos_mean:,.0f} +/- {row.n_pos_std:,.0f}"
        summary["n_pos"] = summary.apply(fmt_n, axis=1)
        summary["n_neg"] = summary.n_neg_mean.apply(lambda x: f"{x:,.0f}")
        summary["tissue_order"] = summary.tissue.apply(lambda t: all_tissues.index(t) if t in all_tissues else 999)
        summary = summary.sort_values(["tissue_order","target"]).drop(columns="tissue_order")
        print(f"\nClassification \u2014 {_cls_labels.get(subset, subset)}:")
        print(summary[["tissue","target","n_pos","n_neg","n_samples"]].to_string(index=False))
else:
    print("no classification data")

# regression: n splice sites per subset
if len(df_reg) > 0:
    reg_tissues = [t for t in tissues if t in df_reg.tissue.unique()]
    for subset, slabel in subset_labels.items():
        sub_reg = df_reg[df_reg.subset == subset]
        sub_reg = sub_reg[sub_reg.tissue.isin(reg_tissues)]
        if len(sub_reg) == 0: continue

        # use classification data for acceptor/donor split
        sub_cls = df_cls[(df_cls.subset == subset)] if len(df_cls) > 0 else pd.DataFrame()
        sub_cls = sub_cls[sub_cls.tissue.isin(reg_tissues)]
        if len(sub_cls) == 0:
            print(f"no classification data for subset={subset}"); continue

        n_df_reg = sub_cls.groupby(["tissue","sample_id","target"]).agg(n_pos=("n_pos","first")).reset_index()
        fig, axes = plt.subplots(1,2,figsize=(12,5))
        for ax, target in zip(axes, ["acceptor","donor"]):
            tsub = n_df_reg[n_df_reg.target == target]
            sns.stripplot(data=tsub, x="tissue", y="n_pos", hue="tissue", order=reg_tissues,
                          palette={t: tissue_colors.get(t,"#999999") for t in reg_tissues},
                          size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)
            for i, tissue in enumerate(reg_tissues):
                vals = tsub[tsub.tissue == tissue].n_pos
                if len(vals) > 0:
                    ax.hlines(vals.mean(), i-0.3, i+0.3, color=tissue_colors.get(tissue,"#999999"), lw=2.5)
            ax.set_xlabel(""); ax.set_ylabel("n splice sites")
            ax.set_xticks(range(len(reg_tissues)))
            ax.set_xticklabels([tissue_display.get(t,t) for t in reg_tissues], rotation=45, ha="right")
            ax.set_title(f"{target.capitalize()} sites")
        fig.suptitle(f"Regression \u2014 {slabel}", y=1.02)
        plt.tight_layout()
        plt.savefig(fig2_sup / f"sample_sizes_reg_{subset}.png", dpi=300, bbox_inches="tight")
        plt.show()

        # summary
        n_summary = sub_cls.groupby(["tissue","target"]).agg(
            n_mean=("n_pos","mean"), n_std=("n_pos","std"), n_samples=("sample_id","nunique")).reset_index()
        n_summary["tissue_order"] = n_summary.tissue.apply(lambda t: reg_tissues.index(t) if t in reg_tissues else 999)
        n_summary = n_summary.sort_values(["tissue_order","target"]).drop(columns="tissue_order")
        print(f"\n{slabel}:")
        for _, row in n_summary.iterrows():
            if pd.isna(row.n_std) or row.n_std == 0:
                print(f"  {row.tissue} ({row.target}): {row.n_mean:,.0f} sites ({int(row.n_samples)} samples)")
            else:
                print(f"  {row.tissue} ({row.target}): {row.n_mean:,.0f} +/- {row.n_std:,.0f} sites ({int(row.n_samples)} samples)")
else:
    print("no regression data")

# Figure 2c: R² swarm plots and classification bar plots

In [ ]:
r2_display = {
    "splaire_ref_ssu":"SPLAIRE","spliceai_cls":"SpliceAI",
    "pangolin_avg_usage":"Pangolin","splicetransformer_avg_tissue":"SpliceTransformer",
}
bar_outputs = ["splaire_ref","spliceai","pangolin_avg_p_splice","splicetransformer"]
bar_display = {"splaire_ref":"SPLAIRE","spliceai":"SpliceAI",
               "pangolin_avg_p_splice":"Pangolin","splicetransformer":"SpliceTransformer"}
r2_to_bar = {"splaire_ref_ssu":"splaire_ref","spliceai_cls":"spliceai",
             "pangolin_avg_usage":"pangolin_avg_p_splice","splicetransformer_avg_tissue":"splicetransformer"}

swarm_subsets = [("ssu_valid","all sites","all"),("ssu_valid_nonzero","SSU > 0","nonzero"),
                 ("ssu_shared","shared sites","shared"),("ssu_shared_nonzero","shared, SSU > 0","shared_nonzero")]

# --- R² swarm plots ---
for subset_name, subset_label, suffix in swarm_subsets:
    sub = df_reg.query("subset == @subset_name").copy()
    sub["base_model"] = sub.output.apply(_get_base_model)
    sub = sub[~sub.base_model.isin(exclude_bases)]
    avail = [o for o in r2_outputs if o in sub.output.unique()]
    sub = sub[sub.output.isin(avail)]
    if len(sub) == 0: continue
    output_order = list(sub.groupby("output").r2.mean().sort_values(ascending=False).index)
    print(f"r2 swarm ({subset_label}): {output_order}")

    # horizontal
    fig, axes = plt.subplots(1,len(tissues_fig1),figsize=(3*len(tissues_fig1),4),sharey=True)
    for col, tissue in enumerate(tissues_fig1):
        ax = axes[col]; tdf = sub[sub.tissue==tissue]
        sns.stripplot(data=tdf,x="output",y="r2",hue="output",order=output_order,
                      palette={o:get_color(o) for o in output_order},
                      size=5,jitter=0.2,alpha=0.7,ax=ax,legend=False)
        for i, out in enumerate(output_order):
            vals = tdf[tdf.output==out].r2
            if len(vals)==0: continue
            ax.hlines(vals.mean(),i-0.3,i+0.3,color=get_color(out),lw=2)
            ax.hlines(vals.median(),i-0.3,i+0.3,color=get_color(out),lw=1.5,ls="--")
        ax.set_xlabel(""); ax.set_ylabel(r"$R^2$" if col==0 else "")
        ax.set_title(tissue_display.get(tissue,tissue))
        ax.set_xticks(range(len(output_order)))
        ax.set_xticklabels([r2_display.get(o,o) for o in output_order],rotation=45,ha="right")
    fig.suptitle(f"R\u00b2 ({subset_label})",y=1.05)
    fig.legend(handles=[Line2D([0],[0],color="gray",lw=2,label="mean"),
                        Line2D([0],[0],color="gray",lw=1.5,ls="--",label="median")],
               loc="upper center",ncol=2,frameon=False,bbox_to_anchor=(0.5,1.02))
    plt.tight_layout()
    plt.savefig(fig2_main / f"r2_swarm_{suffix}.png",dpi=300,bbox_inches="tight"); plt.show()

    # vertical
    fig, axes = plt.subplots(len(tissues_v_order),1,figsize=(4,4*len(tissues_v_order)))
    for row, tissue in enumerate(tissues_v_order):
        ax = axes[row]; tdf = sub[sub.tissue==tissue]
        sns.stripplot(data=tdf,x="output",y="r2",hue="output",order=output_order,
                      palette={o:get_color(o) for o in output_order},
                      size=5,jitter=0.2,alpha=0.7,ax=ax,legend=False)
        for i, out in enumerate(output_order):
            vals = tdf[tdf.output==out].r2
            if len(vals)==0: continue
            ax.hlines(vals.mean(),i-0.3,i+0.3,color=get_color(out),lw=2)
            ax.hlines(vals.median(),i-0.3,i+0.3,color=get_color(out),lw=1.5,ls="--")
        ax.set_xlabel(""); ax.set_ylabel(r"$R^2$")
        ax.set_title(tissue_display.get(tissue,tissue))
        ax.set_xticks(range(len(output_order)))
        if row == len(tissues_v_order)-1:
            ax.set_xticklabels([r2_display.get(o,o) for o in output_order],rotation=45,ha="right")
        else:
            ax.set_xticklabels([])
    fig.suptitle(f"R\u00b2 ({subset_label})",y=1.02)
    fig.legend(handles=[Line2D([0],[0],color="gray",lw=2,label="mean"),
                        Line2D([0],[0],color="gray",lw=1.5,ls="--",label="median")],
               loc="upper center",ncol=2,frameon=False,bbox_to_anchor=(0.5,1.00))
    plt.tight_layout()
    plt.savefig(fig2_main / f"r2_swarm_{suffix}_v.png",dpi=300,bbox_inches="tight"); plt.show()

    # individual per-tissue
    y_min = sub.r2.min() - 0.05; y_max = sub.r2.max() + 0.05
    for tissue in tissues_fig1:
        tdf = sub[sub.tissue==tissue]
        if tdf.empty: continue
        fig_i, ax_i = plt.subplots(1,1,figsize=(3,4))
        sns.stripplot(data=tdf,x="output",y="r2",hue="output",order=output_order,
                      palette={o:get_color(o) for o in output_order},
                      size=5,jitter=0.2,alpha=0.7,ax=ax_i,legend=False)
        for i, out in enumerate(output_order):
            vals = tdf[tdf.output==out].r2
            if len(vals)==0: continue
            ax_i.hlines(vals.mean(),i-0.3,i+0.3,color=get_color(out),lw=2)
            ax_i.hlines(vals.median(),i-0.3,i+0.3,color=get_color(out),lw=1.5,ls="--")
        ax_i.set_xlabel(""); ax_i.set_ylabel(r"$R^2$")
        ax_i.set_title(tissue_display.get(tissue,tissue))
        ax_i.set_xticks(range(len(output_order)))
        ax_i.set_xticklabels([r2_display.get(o,o) for o in output_order],rotation=45,ha="right")
        ax_i.set_ylim(y_min, y_max)
        plt.tight_layout()
        plt.savefig(fig2_main / f"r2_swarm_{suffix}_{tissue}.png",dpi=300,bbox_inches="tight"); plt.show()

# --- Pearson swarm plots ---
for subset_name, subset_label, suffix in swarm_subsets:
    sub = df_reg.query("subset == @subset_name").copy()
    sub["base_model"] = sub.output.apply(_get_base_model)
    sub = sub[~sub.base_model.isin(exclude_bases)]
    avail = [o for o in r2_outputs if o in sub.output.unique()]
    sub = sub[sub.output.isin(avail)]
    if len(sub) == 0: continue
    output_order_p = list(sub.groupby("output").pearson.mean().sort_values(ascending=False).index)
    print(f"pearson swarm ({subset_label}): {output_order_p}")

    fig, axes = plt.subplots(1,len(tissues_fig1),figsize=(3*len(tissues_fig1),4),sharey=True)
    for col, tissue in enumerate(tissues_fig1):
        ax = axes[col]; tdf = sub[sub.tissue==tissue]
        sns.stripplot(data=tdf,x="output",y="pearson",hue="output",order=output_order_p,
                      palette={o:get_color(o) for o in output_order_p},
                      size=5,jitter=0.2,alpha=0.7,ax=ax,legend=False)
        for i, out in enumerate(output_order_p):
            vals = tdf[tdf.output==out].pearson
            if len(vals)==0: continue
            ax.hlines(vals.mean(),i-0.3,i+0.3,color=get_color(out),lw=2)
            ax.hlines(vals.median(),i-0.3,i+0.3,color=get_color(out),lw=1.5,ls="--")
        ax.set_xlabel(""); ax.set_ylabel("Pearson r" if col==0 else "")
        ax.set_title(tissue_display.get(tissue,tissue))
        ax.set_xticks(range(len(output_order_p)))
        ax.set_xticklabels([r2_display.get(o,o) for o in output_order_p],rotation=45,ha="right")
    fig.suptitle(f"Pearson r ({subset_label})",y=1.05)
    fig.legend(handles=[Line2D([0],[0],color="gray",lw=2,label="mean"),
                        Line2D([0],[0],color="gray",lw=1.5,ls="--",label="median")],
               loc="upper center",ncol=2,frameon=False,bbox_to_anchor=(0.5,1.02))
    plt.tight_layout()
    plt.savefig(fig2_main / f"pearson_swarm_{suffix}.png",dpi=300,bbox_inches="tight"); plt.show()

    # vertical
    fig, axes = plt.subplots(len(tissues_v_order),1,figsize=(4,4*len(tissues_v_order)))
    for row, tissue in enumerate(tissues_v_order):
        ax = axes[row]; tdf = sub[sub.tissue==tissue]
        sns.stripplot(data=tdf,x="output",y="pearson",hue="output",order=output_order_p,
                      palette={o:get_color(o) for o in output_order_p},
                      size=5,jitter=0.2,alpha=0.7,ax=ax,legend=False)
        for i, out in enumerate(output_order_p):
            vals = tdf[tdf.output==out].pearson
            if len(vals)==0: continue
            ax.hlines(vals.mean(),i-0.3,i+0.3,color=get_color(out),lw=2)
            ax.hlines(vals.median(),i-0.3,i+0.3,color=get_color(out),lw=1.5,ls="--")
        ax.set_xlabel(""); ax.set_ylabel("Pearson r")
        ax.set_title(tissue_display.get(tissue,tissue))
        ax.set_xticks(range(len(output_order_p)))
        if row == len(tissues_v_order)-1:
            ax.set_xticklabels([r2_display.get(o,o) for o in output_order_p],rotation=45,ha="right")
        else:
            ax.set_xticklabels([])
    fig.suptitle(f"Pearson r ({subset_label})",y=1.02)
    fig.legend(handles=[Line2D([0],[0],color="gray",lw=2,label="mean"),
                        Line2D([0],[0],color="gray",lw=1.5,ls="--",label="median")],
               loc="upper center",ncol=2,frameon=False,bbox_to_anchor=(0.5,1.00))
    plt.tight_layout()
    plt.savefig(fig2_main / f"pearson_swarm_{suffix}_v.png",dpi=300,bbox_inches="tight"); plt.show()

    # individual per-tissue
    y_min = sub.pearson.min() - 0.05; y_max = sub.pearson.max() + 0.05
    for tissue in tissues_fig1:
        tdf = sub[sub.tissue==tissue]
        if tdf.empty: continue
        fig_i, ax_i = plt.subplots(1,1,figsize=(3,4))
        sns.stripplot(data=tdf,x="output",y="pearson",hue="output",order=output_order_p,
                      palette={o:get_color(o) for o in output_order_p},
                      size=5,jitter=0.2,alpha=0.7,ax=ax_i,legend=False)
        for i, out in enumerate(output_order_p):
            vals = tdf[tdf.output==out].pearson
            if len(vals)==0: continue
            ax_i.hlines(vals.mean(),i-0.3,i+0.3,color=get_color(out),lw=2)
            ax_i.hlines(vals.median(),i-0.3,i+0.3,color=get_color(out),lw=1.5,ls="--")
        ax_i.set_xlabel(""); ax_i.set_ylabel("Pearson r")
        ax_i.set_title(tissue_display.get(tissue,tissue))
        ax_i.set_xticks(range(len(output_order_p)))
        ax_i.set_xticklabels([r2_display.get(o,o) for o in output_order_p],rotation=45,ha="right")
        ax_i.set_ylim(y_min, y_max)
        plt.tight_layout()
        plt.savefig(fig2_main / f"pearson_swarm_{suffix}_{tissue}.png",dpi=300,bbox_inches="tight"); plt.show()

# --- bar plots ---
output_order_bar = [r2_to_bar[o] for o in output_order if r2_to_bar.get(o) in bar_outputs]

def make_bar_plots(df, metric_col, ylabel, output_list, display_dict, filename_suffix, tissue_list=None, vertical=False):
    if tissue_list is None: tissue_list = tissues_fig1
    bdf = df.groupby(["tissue","sample_id","output"]).agg(val=(metric_col,"mean")).reset_index()
    bdf["base_model"] = bdf.output.apply(_get_base_model)
    bdf = bdf[~bdf.base_model.isin(exclude_bases)]
    avail = [o for o in output_list if o in bdf.output.unique()]
    bdf = bdf[bdf.output.isin(avail)]
    order = [o for o in output_order_bar if o in avail]
    nt = len(tissue_list)
    if vertical:
        fig, axes = plt.subplots(nt,1,figsize=(4,4*nt))
    else:
        fig, axes = plt.subplots(1,nt,figsize=(3*nt,4),sharey=True)
    if nt==1: axes=[axes]
    _bar_fs = plt.rcParams["font.size"] * 1.25 if vertical else None
    for col, tissue in enumerate(tissue_list):
        ax = axes[col]; tdf = bdf[bdf.tissue==tissue]
        agg = tdf.groupby("output").agg(mean=("val","mean")).reset_index()
        agg = agg.set_index("output").reindex(order).reset_index()
        x = np.arange(len(order))
        bars = ax.bar(x,agg["mean"],color=[get_color(o) for o in order],edgecolor="black",linewidth=0.5,alpha=0.8)
        for bar,val in zip(bars,agg["mean"]):
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()*0.5,f"{val:.2f}",
                    ha="center",va="center",fontweight="bold",rotation=90,color="white",fontsize=_bar_fs)
        ax.set_xlabel(""); ax.set_ylabel(ylabel)
        ax.set_title(tissue_display.get(tissue,tissue))
        ax.set_xticks(x)
        if not vertical or col == nt-1:
            ax.set_xticklabels([display_dict.get(o,o) for o in order],rotation=45,ha="right")
        else:
            ax.set_xticklabels([])
        ax.set_ylim(0,1.1)
    plt.tight_layout(); plt.savefig(fig2_main / f"{filename_suffix}.png",dpi=300); plt.show()

make_bar_plots(df_cls.query("subset == 'all'"),"topk","Top-k",bar_outputs,bar_display,"topk_bars")
make_bar_plots(df_cls.query("subset == 'all'"),"auprc","AUPRC",bar_outputs,bar_display,"auprc_bars")
make_bar_plots(df_cls.query("subset == 'all'"),"topk","Top-k",bar_outputs,bar_display,"topk_bars_v",vertical=True)
make_bar_plots(df_cls.query("subset == 'all'"),"auprc","AUPRC",bar_outputs,bar_display,"auprc_bars_v",vertical=True)

# --- combined figure: AUPRC bars | binned AUPRC | binned MSE | R² swarm ---
tissues_comb = [m for d in _desired_v if (m := _match_tissue(d, tissues_fig1)) is not None]
if not tissues_comb: tissues_comb = tissues_fig1
nt = len(tissues_comb)

_orig_params = {k: plt.rcParams[k] for k in ["font.size","axes.titlesize","axes.labelsize",
                "xtick.labelsize","ytick.labelsize","legend.fontsize","legend.title_fontsize","figure.titlesize"]}
_scale = 1.4
for k, v in _orig_params.items(): plt.rcParams[k] = v * _scale
_gap_01,_gap_12,_gap_23 = 0.35,0.65,0.65
col_titles = ["AUPRC","Binned AUPRC","Binned MSE",r"$R^2$"]

_bar_df = df_cls.query("subset == 'all'").copy()
_bar_df = _bar_df.groupby(["tissue","sample_id","output"]).agg(val=("auprc","mean")).reset_index()
_bar_df["base_model"] = _bar_df.output.apply(_get_base_model)
_bar_df = _bar_df[~_bar_df.base_model.isin(exclude_bases)]
_bar_avail = [o for o in bar_outputs if o in _bar_df.output.unique()]
_bar_df = _bar_df[_bar_df.output.isin(_bar_avail)]
_bar_order = [o for o in output_order_bar if o in _bar_avail]

_r2_sub = df_reg.query("subset == 'ssu_valid'").copy()
_r2_sub["base_model"] = _r2_sub.output.apply(_get_base_model)
_r2_sub = _r2_sub[~_r2_sub.base_model.isin(exclude_bases)]
_r2_avail = [o for o in r2_outputs if o in _r2_sub.output.unique()]
_r2_sub = _r2_sub[_r2_sub.output.isin(_r2_avail)]
_r2_order = list(_r2_sub.groupby("output").r2.mean().sort_values(ascending=False).index)

_mc = "mae" if "mae" in df_bin_reg.columns else "mse"
_bin_labels = [f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)]
_mse_vals = []
for t in tissues_comb:
    sm = df_bin_reg.query("tissue == @t & subset == 'ssu_valid_nonzero'")
    if len(sm) > 0:
        av = [o for o in output_order_spec if o in sm.output.unique()]
        sm = sm[sm.output.isin(av)]
        _mse_vals.extend(sm.groupby(["output","bin"]).agg(metric=(_mc,"mean")).reset_index()["metric"].dropna().tolist())
_mse_ylim = (0, max(_mse_vals)*1.1) if _mse_vals else (0,1)
_r2_vals = []
for t in tissues_comb:
    tr = _r2_sub[_r2_sub.tissue==t]
    if len(tr) > 0: _r2_vals.extend(tr.r2.tolist())
_r2_ylim = (min(_r2_vals)-0.02, max(_r2_vals)+0.02) if _r2_vals else (0,1)
_sec_fs = plt.rcParams["font.size"]*0.75

def _plot_row(fig,gs,row,tissue,show_titles=False,show_xlabels=False):
    _dc = [0,2,4,6]
    # col 0: AUPRC bars
    ax = fig.add_subplot(gs[row,_dc[0]])
    tdf = _bar_df[_bar_df.tissue==tissue]
    agg = tdf.groupby("output").agg(mean=("val","mean")).reset_index()
    agg = agg.set_index("output").reindex(_bar_order).reset_index()
    x = np.arange(len(_bar_order))
    bars = ax.bar(x,agg["mean"],color=[get_color(o) for o in _bar_order],edgecolor="black",linewidth=0.5,alpha=0.8)
    for b,v in zip(bars,agg["mean"]):
        ax.text(b.get_x()+b.get_width()/2,b.get_height()*0.5,f"{v:.2f}",ha="center",va="center",
                fontweight="bold",rotation=90,color="white",fontsize=plt.rcParams["font.size"]*1.25)
    ax.set_ylim(0,1.1); ax.set_xticks(x)
    ax.set_xticklabels([bar_display.get(o,o) for o in _bar_order] if show_xlabels else [],rotation=45,ha="right")
    ax.set_ylabel("AUPRC")
    ax.text(-0.35,0.5,tissue_display.get(tissue,tissue),transform=ax.transAxes,ha="right",va="center",
            fontweight="bold",fontsize=plt.rcParams["axes.labelsize"],rotation=90)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    if show_titles: ax.set_title(col_titles[0],pad=15)

    # col 1: binned AUPRC
    ax = fig.add_subplot(gs[row,_dc[1]])
    sub = df_bin_ratio.query("tissue==@tissue & subset=='ssu_valid_nonzero'").copy()
    if len(sub) > 0:
        avail = [o for o in cls_order if o in sub.output.unique()]
        sub = sub[sub.output.isin(avail)]
        ag = sub.groupby(["output","bin"]).agg(auprc=("auprc","mean"),n_pos=("n_pos","mean")).reset_index()
        ct = ag[ag.output==avail[0]][["bin","n_pos"]].sort_values("bin")
        ax2 = ax.twinx()
        ax2.bar(ct["bin"],ct["n_pos"],width=0.5,color=GRAY_FAINT,alpha=0.4,zorder=1)
        ax2.tick_params(axis="y",labelcolor=GRAY_MED); ax2.set_yscale("log")
        ax2.set_ylim(ct["n_pos"].min()*0.5,ct["n_pos"].max()*3)
        ax2.set_ylabel("Splice Sites",color=GRAY_MED,fontsize=_sec_fs)
        for out in avail:
            od = ag[ag.output==out].sort_values("bin")
            ax.plot(od["bin"],od["auprc"],marker="o",color=get_color(out),lw=2,markersize=4,zorder=3)
        ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
    ax.set_ylim(0,1); ax.set_ylabel("AUPRC"); ax.set_xticks(range(10))
    if show_xlabels: ax.set_xticklabels(_bin_labels,rotation=45,ha="right"); ax.set_xlabel("SSU bin")
    else: ax.set_xticklabels([])
    if show_titles: ax.set_title(col_titles[1],pad=15)

    # col 2: binned MSE
    ax = fig.add_subplot(gs[row,_dc[2]])
    sub = df_bin_reg.query("tissue==@tissue & subset=='ssu_valid_nonzero'").copy()
    if len(sub) > 0:
        avail = [o for o in output_order_spec if o in sub.output.unique()]
        sub = sub[sub.output.isin(avail)]
        ag = sub.groupby(["output","bin"]).agg(metric=(_mc,"mean"),n=("n","mean")).reset_index()
        ct = ag[ag.output==avail[0]][["bin","n"]].sort_values("bin")
        ax2 = ax.twinx()
        ax2.bar(ct["bin"],ct["n"],width=0.5,color=GRAY_FAINT,alpha=0.4,zorder=1)
        ax2.tick_params(axis="y",labelcolor=GRAY_MED); ax2.set_yscale("log")
        ax2.set_ylim(ct["n"].min()*0.5,ct["n"].max()*3)
        ax2.set_ylabel("Splice Sites",color=GRAY_MED,fontsize=_sec_fs)
        for out in avail:
            od = ag[ag.output==out].sort_values("bin")
            ax.plot(od["bin"],od["metric"],marker="o",color=get_color(out),lw=2,markersize=4,zorder=3,
                    ls="--" if out in reg_outputs else "-")
        ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
    ax.set_ylabel("MSE"); ax.set_ylim(*_mse_ylim); ax.set_xticks(range(10))
    if show_xlabels: ax.set_xticklabels(_bin_labels,rotation=45,ha="right"); ax.set_xlabel("SSU bin")
    else: ax.set_xticklabels([])
    if show_titles: ax.set_title(col_titles[2],pad=15)

    # col 3: R² swarm
    ax = fig.add_subplot(gs[row,_dc[3]])
    tdf = _r2_sub[_r2_sub.tissue==tissue]
    if len(tdf) > 0:
        sns.stripplot(data=tdf,x="output",y="r2",hue="output",order=_r2_order,
                      palette={o:get_color(o) for o in _r2_order},size=4,jitter=0.2,alpha=0.7,ax=ax,legend=False)
        for i,out in enumerate(_r2_order):
            vals = tdf[tdf.output==out].r2
            if len(vals)==0: continue
            ax.hlines(vals.mean(),i-0.3,i+0.3,color=get_color(out),lw=2)
            ax.hlines(vals.median(),i-0.3,i+0.3,color=get_color(out),lw=1.5,ls="--")
    ax.set_xlabel(""); ax.set_ylabel(r"$R^2$"); ax.set_ylim(*_r2_ylim)
    ax.set_xticks(range(len(_r2_order)))
    if show_xlabels: ax.set_xticklabels([r2_display.get(o,o) for o in _r2_order],rotation=45,ha="right")
    else: ax.set_xticklabels([])
    if show_titles: ax.set_title(col_titles[3],pad=15)

fig = plt.figure(figsize=(28,4.5*nt))
gs = fig.add_gridspec(nt,7,width_ratios=[1,_gap_01,1.5,_gap_12,1.5,_gap_23,1],hspace=0.35,wspace=0.05)
for row, tissue in enumerate(tissues_comb):
    _plot_row(fig,gs,row,tissue,show_titles=(row==0),show_xlabels=(row==nt-1))
for k,v in _orig_params.items(): plt.rcParams[k] = v
plt.savefig(fig2_main / "combined_vertical.png",dpi=300,bbox_inches="tight"); plt.show()
print(f"saved {fig2_main}/combined_vertical.png")

# individual tissue combined figures
for tissue in tissues_comb:
    for k,v in _orig_params.items(): plt.rcParams[k] = v * _scale
    fig_t = plt.figure(figsize=(28,5.5))
    gs_t = fig_t.add_gridspec(1,7,width_ratios=[1,_gap_01,1.5,_gap_12,1.5,_gap_23,1],wspace=0.05)
    _plot_row(fig_t,gs_t,0,tissue,show_titles=True,show_xlabels=True)
    for k,v in _orig_params.items(): plt.rcParams[k] = v
    tname = tissue.lower().replace(" ","_").replace("-","_")
    fig_t.savefig(fig2_main / f"combined_{tname}.png",dpi=300,bbox_inches="tight"); plt.show()
    print(f"saved {fig2_main}/combined_{tname}.png")

# standalone legend
from collections import OrderedDict
_all_outputs_leg = [o for o in output_order_spec
                    if o in df_bin_reg[df_bin_reg.subset=="ssu_valid_nonzero"].output.unique().tolist()]
_legend_groups = OrderedDict()
for o in _all_outputs_leg:
    bm = _get_base_model(o)
    if bm not in _legend_groups: _legend_groups[bm] = []
    ls = "--" if o in reg_outputs else "-"
    _legend_groups[bm].append((o, ls))
_base_display = {"splaire_ref":"SPLAIRE","pangolin":"Pangolin","splicetransformer":"SpliceTransformer","spliceai":"SpliceAI"}
_lh, _ll = [], []
for i,(bm,entries) in enumerate(_legend_groups.items()):
    if i > 0: _lh.append(Line2D([0],[0],color="none",lw=0)); _ll.append("")
    for o, ls in entries:
        name = _base_display.get(bm, bm)
        _lh.append(Line2D([0],[0],color=get_color(o),lw=2.5,ls=ls,marker="o",markersize=8))
        _ll.append(f"{name} - SSU" if ls=="--" else name)
fig_leg, ax_leg = plt.subplots(figsize=(3,0.5*len(_ll)))
ax_leg.axis("off")
ax_leg.legend(handles=_lh,labels=_ll,loc="center",frameon=True,edgecolor="lightgray",fancybox=True,
              fontsize=plt.rcParams["font.size"]*_scale,handlelength=3,handleheight=1.5,labelspacing=0.6)
fig_leg.savefig(fig2_main / "combined_legend.png",dpi=300,bbox_inches="tight"); plt.show()

# Pangolin Test Set: Bar Plots by Tissue

In [ ]:
# pangolin test set bar plots (same layout as figs.ipynb)

if len(df_pang_cls) == 0:
    print("no pangolin test data loaded — skipping bar plots")
else:
    bar_outputs = ["splaire_ref", "spliceai", "pangolin_avg_p_splice", "splicetransformer"]
    bar_display = {"splaire_ref": "SPLAIRE", "spliceai": "SpliceAI",
                   "pangolin_avg_p_splice": "Pangolin", "splicetransformer": "SpliceTransformer"}
    r2_display_pang = {"splaire_ref_ssu": "SPLAIRE", "spliceai_cls": "SpliceAI",
                       "pangolin_avg_usage": "Pangolin", "splicetransformer_avg_tissue": "SpliceTransformer"}

    pang_cls_order = [o for o in bar_outputs if o in df_pang_cls.output.unique()]
    pang_reg_order = [o for o in r2_outputs if o in df_pang_reg.output.unique()]
    print(f"cls outputs: {pang_cls_order}")
    print(f"reg outputs: {pang_reg_order}")

    def make_pang_bar(df, metric, ylabel, output_order, display_dict, tissues, suffix, vertical=False):
        """bar plots for pangolin test (one value per tissue x output)"""
        sub = df[df.tissue.isin(tissues) & df.output.isin(output_order)].copy()
        nt = len(tissues)
        if vertical:
            fig, axes = plt.subplots(nt, 1, figsize=(4, 4 * nt))
        else:
            fig, axes = plt.subplots(1, nt, figsize=(3 * nt, 4), sharey=True)
        if nt == 1: axes = [axes]
        for col, tissue in enumerate(tissues):
            ax = axes[col]
            tdf = sub[sub.tissue == tissue].set_index("output").reindex(output_order)
            x = np.arange(len(output_order))
            colors = [get_color(o) for o in output_order]
            vals = tdf[metric].values
            bars = ax.bar(x, vals, color=colors, edgecolor="black", linewidth=0.5, alpha=0.8)
            for bar, v in zip(bars, vals):
                if np.isfinite(v):
                    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                            f"{v:.2f}", ha="center", va="center", fontweight="bold",
                            rotation=90, color="white", fontsize=ANNOT_SIZE)
            ax.set_xlabel("")
            ax.set_ylabel(ylabel if col == 0 or vertical else "")
            ax.set_title(pang_tissue_display.get(tissue, tissue))
            ax.set_xticks(x)
            if not vertical or col == nt - 1:
                ax.set_xticklabels([display_dict.get(o, o) for o in output_order],
                                   rotation=45, ha="right")
            else:
                ax.set_xticklabels([])
            if metric in {"auprc", "topk", "f1"}: ax.set_ylim(0, 1.1)
        plt.tight_layout()
        plt.savefig(fig2_main / f"pang_{suffix}.png", dpi=300, bbox_inches="tight")
        plt.show()

    # classification bar plots
    print("\n=== pangolin test classification ===")
    for metric, ylabel in [("topk", "Top-k"), ("auprc", "AUPRC"), ("f1", "F1")]:
        make_pang_bar(df_pang_cls, metric, ylabel, pang_cls_order, bar_display,
                      pang_test_tissues, f"{metric}_bars")
        make_pang_bar(df_pang_cls, metric, ylabel, pang_cls_order, bar_display,
                      pang_test_tissues, f"{metric}_bars_v", vertical=True)

    # regression bar plots
    if len(df_pang_reg) > 0:
        print("\n=== pangolin test regression ===")
        for subset_name, suffix in [("ssu_valid", "all"), ("ssu_valid_nonzero", "nonzero")]:
            sub_reg = df_pang_reg[df_pang_reg.subset == subset_name]
            if sub_reg.empty: continue
            subset_label = "all sites" if suffix == "all" else "SSU > 0"
            for metric, ylabel in [("r2", r"$R^2$"), ("pearson", "Pearson r")]:
                make_pang_bar(sub_reg, metric, f"{ylabel} ({subset_label})",
                              pang_reg_order, r2_display_pang,
                              pang_test_tissues, f"{metric}_bars_{suffix}")
                make_pang_bar(sub_reg, metric, f"{ylabel} ({subset_label})",
                              pang_reg_order, r2_display_pang,
                              pang_test_tissues, f"{metric}_bars_{suffix}_v", vertical=True)

    # individual per-tissue plots
    print("\n=== individual per-tissue ===")
    for tissue in pang_test_tissues:
        for metric, ylabel in [("topk", "Top-k"), ("auprc", "AUPRC"), ("f1", "F1")]:
            tdf = df_pang_cls[(df_pang_cls.tissue == tissue) &
                              df_pang_cls.output.isin(pang_cls_order)]
            if tdf.empty: continue
            tdf = tdf.set_index("output").reindex(pang_cls_order)
            fig, ax = plt.subplots(1, 1, figsize=(3, 4))
            x = np.arange(len(pang_cls_order))
            vals = tdf[metric].values
            bars = ax.bar(x, vals, color=[get_color(o) for o in pang_cls_order],
                         edgecolor="black", linewidth=0.5, alpha=0.8)
            for bar, v in zip(bars, vals):
                if np.isfinite(v):
                    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                            f"{v:.2f}", ha="center", va="center", fontweight="bold",
                            rotation=90, color="white")
            ax.set_ylabel(ylabel)
            ax.set_title(pang_tissue_display.get(tissue, tissue))
            ax.set_xticks(x)
            ax.set_xticklabels([bar_display.get(o, o) for o in pang_cls_order],
                               rotation=45, ha="right")
            ax.set_ylim(0, 1.1)
            plt.tight_layout()
            plt.savefig(fig2_main / f"pang_{metric}_{tissue}.png", dpi=300, bbox_inches="tight")
            plt.show()

        for subset_name, suffix in [("ssu_valid", "all"), ("ssu_valid_nonzero", "nonzero")]:
            sub_reg = df_pang_reg[(df_pang_reg.subset == subset_name) &
                                  (df_pang_reg.tissue == tissue) &
                                  df_pang_reg.output.isin(pang_reg_order)]
            if sub_reg.empty: continue
            sub_reg = sub_reg.set_index("output").reindex(pang_reg_order)
            for metric, ylabel in [("r2", r"$R^2$"), ("pearson", "Pearson r")]:
                fig, ax = plt.subplots(1, 1, figsize=(3, 4))
                x = np.arange(len(pang_reg_order))
                vals = sub_reg[metric].values
                bars = ax.bar(x, vals, color=[get_color(o) for o in pang_reg_order],
                             edgecolor="black", linewidth=0.5, alpha=0.8)
                for bar, v in zip(bars, vals):
                    if np.isfinite(v):
                        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                                f"{v:.2f}", ha="center", va="center", fontweight="bold",
                                rotation=90, color="white")
                ax.set_ylabel(ylabel)
                ax.set_title(pang_tissue_display.get(tissue, tissue))
                ax.set_xticks(x)
                ax.set_xticklabels([r2_display_pang.get(o, o) for o in pang_reg_order],
                                   rotation=45, ha="right")
                plt.tight_layout()
                plt.savefig(fig2_main / f"pang_{metric}_{suffix}_{tissue}.png",
                            dpi=300, bbox_inches="tight")
                plt.show()

    print(f"\nsaved pangolin test bar plots to {fig2_main}")

# Calibration: predicted SSU vs actual SSU
bins predictions into 20 equal-width bins, computes mean actual SSU per bin.
perfect calibration = diagonal.

In [ ]:
import sys, pickle, time
from concurrent.futures import ProcessPoolExecutor, as_completed

sys.path.insert(0, "src")

# model file suffixes for parquet loading
_models = {
    "_splaire_ref": ("splaire_ref", ["acceptor", "donor", "ssu"]),
    "_splaire_var": ("splaire_var", ["acceptor", "donor", "ssu"]),
    "_sa": ("spliceai", ["acceptor", "donor"]),
    "_pang": ("pangolin", [
        "heart_p_splice", "heart_usage", "liver_p_splice", "liver_usage",
        "brain_p_splice", "brain_usage", "testis_p_splice", "testis_usage",
        "avg_p_splice", "avg_usage",
    ]),
    "_spt": ("splicetransformer", [
        "acceptor", "donor",
        "adipose", "blood", "blood_vessel", "brain", "colon", "heart", "kidney",
        "liver", "lung", "muscle", "nerve", "small_intestine", "skin", "spleen", "stomach",
        "avg_tissue",
    ]),
}

def load_predictions(pred_dir):
    # load ground truth and model predictions from parquet files
    files = {}
    for pq in pred_dir.glob("*.parquet"):
        for suffix, (name, _cols) in _models.items():
            if suffix in pq.stem:
                files[name] = pq
                break
    any_df = pd.read_parquet(next(iter(files.values())))
    truth = {
        "acceptor": any_df["y_acceptor"].values.astype(np.float32),
        "donor": any_df["y_donor"].values.astype(np.float32),
        "ssu": any_df["y_ssu"].values.astype(np.float32),
    }
    name_to_cols = {name: cols for _suffix, (name, cols) in _models.items()}
    preds = {}
    for name, path in files.items():
        df = pd.read_parquet(path)
        cols = name_to_cols[name]
        preds[name] = {c: df[c].values.astype(np.float32) for c in cols}
    return truth, preds

def add_splaire_avg(preds):
    # average splaire_ref and splaire_var predictions
    if "splaire_ref" in preds and "splaire_var" in preds:
        ref, var = preds["splaire_ref"], preds["splaire_var"]
        preds["splaire_avg"] = {k: (ref[k] + var[k]) / 2.0 for k in ["acceptor", "donor", "ssu"]}

cache_path = Path("dists/data/calibration_cache_v2.pkl")
pred_base = Path("/scratch/runyan.m/sphaec_out")
n_cal_bins = 20
cal_edges = np.linspace(0, 1, n_cal_bins + 1)
SENTINEL = 777.0
N_WORKERS = 4
cal_tissues = [t for t in tissues_fig1 if t != "whole_blood"]

def _process_individual(pred_dir, cal_edges, n_cal_bins, SENTINEL):
    # load one individual and compute calibration bins
    import numpy as np
    import pandas as pd

    sid = pred_dir.name
    t0 = time.time()

    # inline parquet loading (avoids import path issues in subprocess)
    _mdls = {
        "_splaire_ref": ("splaire_ref", ["acceptor", "donor", "ssu"]),
        "_splaire_var": ("splaire_var", ["acceptor", "donor", "ssu"]),
        "_sa": ("spliceai", ["acceptor", "donor"]),
        "_pang": ("pangolin", [
            "heart_p_splice", "heart_usage", "liver_p_splice", "liver_usage",
            "brain_p_splice", "brain_usage", "testis_p_splice", "testis_usage",
            "avg_p_splice", "avg_usage",
        ]),
        "_spt": ("splicetransformer", [
            "acceptor", "donor",
            "adipose", "blood", "blood_vessel", "brain", "colon", "heart", "kidney",
            "liver", "lung", "muscle", "nerve", "small_intestine", "skin", "spleen", "stomach",
            "avg_tissue",
        ]),
    }
    files = {}
    for pq in pred_dir.glob("*.parquet"):
        for suffix, (name, _cols) in _mdls.items():
            if suffix in pq.stem:
                files[name] = pq
                break
    any_df = pd.read_parquet(next(iter(files.values())))
    truth = {
        "acceptor": any_df["y_acceptor"].values.astype(np.float32),
        "donor": any_df["y_donor"].values.astype(np.float32),
        "ssu": any_df["y_ssu"].values.astype(np.float32),
    }
    name_to_cols = {name: cols for _suffix, (name, cols) in _mdls.items()}
    preds = {}
    for name, path in files.items():
        df = pd.read_parquet(path)
        cols = name_to_cols[name]
        preds[name] = {c: df[c].values.astype(np.float32) for c in cols}
    # add splaire_avg
    if "splaire_ref" in preds and "splaire_var" in preds:
        ref, var = preds["splaire_ref"], preds["splaire_var"]
        preds["splaire_avg"] = {k: (ref[k]+var[k])/2.0 for k in ["acceptor","donor","ssu"]}
    load_time = time.time() - t0

    is_acc = truth["acceptor"] == 1; is_don = truth["donor"] == 1; ssu = truth["ssu"]
    valid = (ssu != SENTINEL) & (ssu > 0)
    site_idx = np.flatnonzero((is_acc | is_don) & valid)
    acc_idx = np.flatnonzero(is_acc & valid)
    don_idx = np.flatnonzero(is_don & valid)
    y_true_all = ssu[site_idx].astype(np.float32)

    sample_cal = {}
    for model, p in preds.items():
        # cls output: matched acc/don predictions
        if "acceptor" in p and "donor" in p:
            col = f"{model}_cls"
            y_pred = np.concatenate([p["acceptor"][acc_idx], p["donor"][don_idx]]).astype(np.float32)
            y_true_cls = np.concatenate([ssu[acc_idx], ssu[don_idx]]).astype(np.float32)
            bi = np.clip(np.digitize(y_pred, cal_edges) - 1, 0, n_cal_bins - 1)
            pm, tm, ct = [], [], []
            for b in range(n_cal_bins):
                m = bi == b; n = m.sum(); ct.append(int(n))
                pm.append(float(y_pred[m].mean()) if n > 0 else np.nan)
                tm.append(float(y_true_cls[m].mean()) if n > 0 else np.nan)
            sample_cal[col] = {"pred_mean": pm, "true_mean": tm, "count": ct}

        # single-score outputs
        for key, arr in p.items():
            if key in ["acceptor", "donor", "neither", "splice"]: continue
            col = f"{model}_{key}"
            y_pred = arr[site_idx].astype(np.float32)
            bi = np.clip(np.digitize(y_pred, cal_edges) - 1, 0, n_cal_bins - 1)
            pm, tm, ct = [], [], []
            for b in range(n_cal_bins):
                m = bi == b; n = m.sum(); ct.append(int(n))
                pm.append(float(y_pred[m].mean()) if n > 0 else np.nan)
                tm.append(float(y_true_all[m].mean()) if n > 0 else np.nan)
            sample_cal[col] = {"pred_mean": pm, "true_mean": tm, "count": ct}
    return sid, sample_cal, len(site_idx), len(sample_cal), time.time()-t0, load_time

if cache_path.exists():
    with open(cache_path, "rb") as f: cal_data = pickle.load(f)
    print(f"loaded cached calibration from {cache_path}")
else:
    cal_data = {}

needs = [t for t in cal_tissues if t not in cal_data]
if needs:
    print(f"computing: {needs}")
    for ti, tissue in enumerate(needs):
        t0t = time.time()
        pred_root = pred_base / tissue / "ml_out_var" / "predictions"
        if not pred_root.exists():
            print(f"{tissue}: no predictions at {pred_root}"); continue
        pred_dirs = sorted(d for d in pred_root.iterdir() if d.is_dir())
        print(f"[{ti+1}/{len(needs)}] {tissue}: {len(pred_dirs)} individuals")
        tissue_cal = {}
        done = 0
        with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
            futs = {pool.submit(_process_individual, d, cal_edges, n_cal_bins, SENTINEL): d for d in pred_dirs}
            for fut in as_completed(futs):
                done += 1
                try:
                    sid, sc, ns, no, el, lt = fut.result()
                    tissue_cal[sid] = sc
                    print(f"  [{done}/{len(pred_dirs)}] {sid}: {ns:,} sites, {el:.1f}s")
                except Exception as e:
                    print(f"  [{done}/{len(pred_dirs)}] FAILED: {e}")
        cal_data[tissue] = tissue_cal
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cache_path, "wb") as f: pickle.dump(cal_data, f)
        print(f"{tissue}: {len(tissue_cal)} individuals, {(time.time()-t0t)/60:.1f}min")
else:
    print("all cached")

for tissue in cal_data:
    outputs = set()
    for sid in cal_data[tissue]: outputs.update(cal_data[tissue][sid].keys())
    print(f"{tissue}: {len(cal_data[tissue])} individuals, {len(outputs)} outputs")

In [ ]:
cal_cls_outputs = ["spliceai_cls","splaire_var_cls","pangolin_avg_p_splice","splicetransformer_cls"]
cal_ssu_outputs = ["splaire_var_ssu","pangolin_avg_usage","splicetransformer_avg_tissue"]
cal_display = {
    "splaire_var_ssu":"SPLAIRE SSU","splaire_var_cls":"SPLAIRE CLS",
    "pangolin_avg_usage":"Pangolin Avg Usage","pangolin_avg_p_splice":"Pangolin Avg CLS",
    "splicetransformer_avg_tissue":"SpliceTransformer Avg Tissue",
    "splicetransformer_cls":"SpliceTransformer CLS","spliceai_cls":"SpliceAI CLS",
}

def _cal_curve(tdata, output):
    # weighted-mean calibration curve across individuals
    all_pred, all_true, all_count = [], [], []
    for sid, scal in tdata.items():
        if output not in scal: continue
        all_pred.append(scal[output]["pred_mean"])
        all_true.append(scal[output]["true_mean"])
        all_count.append(scal[output]["count"])
    if not all_pred: return None, None
    pred_arr = np.array(all_pred); true_arr = np.array(all_true)
    count_arr = np.array(all_count, dtype=float)
    with np.errstate(invalid="ignore"):
        count_arr[np.isnan(pred_arr)] = 0
        w_sum = count_arr.sum(axis=0)
        pred_mean = np.where(w_sum > 0, np.nansum(pred_arr*count_arr, axis=0) / w_sum, np.nan)
        true_mean = np.where(w_sum > 0, np.nansum(true_arr*count_arr, axis=0) / w_sum, np.nan)
    valid = np.isfinite(pred_mean) & np.isfinite(true_mean)
    return pred_mean[valid], true_mean[valid]

nt = len(cal_tissues); pw = max(7, 6*nt)

# CLS calibration
fig, axes = plt.subplots(1, nt, figsize=(pw, 7), sharey=True)
if nt == 1: axes = [axes]
for ti, tissue in enumerate(cal_tissues):
    ax = axes[ti]; tdata = cal_data.get(tissue, {})
    for output in cal_cls_outputs:
        pm, tm = _cal_curve(tdata, output)
        if pm is None: continue
        ax.plot(pm, tm, marker="o", markersize=6, color=get_color(output),
                lw=2.5, label=cal_display.get(output, output))
    ax.plot([0,1], [0,1], "k--", alpha=0.3, lw=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("Predicted Score", fontsize=14)
    if ti == 0: ax.set_ylabel("Actual SSU", fontsize=14)
    ax.set_title(tissue_display.get(tissue, tissue), fontsize=15, fontweight="bold")
    ax.grid(alpha=0.15); ax.legend(fontsize=12, loc="upper left", framealpha=0.9)
fig.suptitle("Calibration \u2014 Classification Outputs", fontsize=17, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(fig2_main / "calibration_cls.png", dpi=300, bbox_inches="tight"); plt.show()
print(f"saved {fig2_main}/calibration_cls.png")

# SSU/usage calibration
fig, axes = plt.subplots(1, nt, figsize=(pw, 7), sharey=True)
if nt == 1: axes = [axes]
for ti, tissue in enumerate(cal_tissues):
    ax = axes[ti]; tdata = cal_data.get(tissue, {})
    for output in cal_ssu_outputs:
        pm, tm = _cal_curve(tdata, output)
        if pm is None: continue
        ax.plot(pm, tm, marker="o", markersize=6, color=get_color(output),
                lw=2.5, ls="--", label=cal_display.get(output, output))
    ax.plot([0,1], [0,1], "k--", alpha=0.3, lw=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("Predicted SSU", fontsize=14)
    if ti == 0: ax.set_ylabel("Actual SSU", fontsize=14)
    ax.set_title(tissue_display.get(tissue, tissue), fontsize=15, fontweight="bold")
    ax.grid(alpha=0.15); ax.legend(fontsize=12, loc="upper left", framealpha=0.9)
fig.suptitle("Calibration \u2014 SSU/Usage Outputs", fontsize=17, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(fig2_main / "calibration_ssu.png", dpi=300, bbox_inches="tight"); plt.show()
print(f"saved {fig2_main}/calibration_ssu.png")

## Supplemental: All-Output Heatmaps

Heatmaps showing ALL model outputs (not just best per family), with best-per-model and best-overall annotations.

In [ ]:
# heatmaps showing ALL outputs (not just best per base model)
# red solid = best overall per tissue, orange dashed = best per base model family
# includes canonical benchmarks

lower_better = {"mse", "mae"}

# tissue lists for heatmaps
_heatmap_tissues = list(tissues_fig1)
_heatmap_tissues_cls = _heatmap_tissues + [t for t in gencode_tissues if t in df_cls.tissue.unique()]

_output_display = {
    "sphaec_ref": "SPLAIRE CLS",
    "sphaec_ref_ssu": "SPLAIRE SSU",
    "spliceai": "SpliceAI CLS",
    "splicetransformer": "SpliceTransformer CLS",
    "splicetransformer_avg_tissue": "SpliceTransformer Avg Tissue",
}

def _output_name(code):
    """pretty name for model output"""
    if code in _output_display:
        return _output_display[code]
    if code.startswith("pangolin_"):
        rest = code.replace("pangolin_", "")
        if rest.endswith("_p_splice"):
            tissue = rest.replace("_p_splice", "").replace("_", " ").title()
            return f"Pangolin {tissue} CLS"
        if rest.endswith("_usage"):
            tissue = rest.replace("_usage", "").replace("_", " ").title()
            return f"Pangolin {tissue} Usage"
        return f"Pangolin {rest.replace('_', ' ').title()}"
    if code.startswith("splicetransformer_"):
        tissue = code.replace("splicetransformer_", "").replace("_", " ").title()
        return f"SpliceTransformer {tissue}"
    return code


def plot_heatmap_all_outputs(data, metric, title_suffix="", fname_suffix="", out_dir=None, tissue_list=None):
    """heatmap with all outputs, highlighting best per base and best overall"""
    if tissue_list is None:
        tissue_list = all_tissues
    
    sub = data[data.metric == metric]
    if len(sub) == 0:
        print(f"no data for {metric}")
        return
    
    pivot = sub.pivot(index="output", columns="tissue", values="score")
    pivot = pivot[[t for t in tissue_list if t in pivot.columns]]
    
    # sort by mean performance (best at top)
    ascending = metric in lower_better
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=ascending).index]
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.4)))
    im = ax.imshow(pivot.values, cmap="YlGnBu", aspect="auto")
    
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([tissue_display.get(t, t) for t in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([_output_name(o) for o in pivot.index])
    
    # add vertical line to separate expression vs gencode
    n_expr = len([t for t in pivot.columns if t in set(tissues_fig1)])
    if n_expr < len(pivot.columns):
        ax.axvline(x=n_expr - 0.5, color="white", lw=2, ls="--")
    
    # build output -> base model mapping
    output_to_base = {out: _get_base_model(out) for out in pivot.index}
    base_models = set(output_to_base.values()) - {None}
    
    for j in range(len(pivot.columns)):
        col_vals = pivot.iloc[:, j]
        
        # best per base model (orange dashed)
        for base in base_models:
            base_outputs = [out for out, b in output_to_base.items() if b == base]
            base_vals = col_vals.loc[base_outputs]
            best_out = base_vals.idxmin() if metric in lower_better else base_vals.idxmax()
            i = list(pivot.index).index(best_out)
            rect = Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="orange", lw=1.5, ls="--")
            ax.add_patch(rect)
        
        # best overall (red solid)
        best_i = col_vals.idxmin() if metric in lower_better else col_vals.idxmax()
        i = list(pivot.index).index(best_i)
        rect = Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="red", lw=2.5)
        ax.add_patch(rect)
    
    # add values
    vmin, vmax = pivot.values.min(), pivot.values.max()
    mid = (vmin + vmax) / 2
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.iloc[i, j]
            color = "white" if v > mid else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color=color, fontsize=ANNOT_SIZE)
    
    plt.colorbar(im, ax=ax, shrink=0.5)
    ax.set_title(f"{metric.upper()}{title_suffix}")
    
    # legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="none", edgecolor="red", lw=2.5, label="best overall"),
        Patch(facecolor="none", edgecolor="orange", lw=1.5, ls="--", label="best per model"),
    ]
    ax.legend(handles=legend_elements, loc="upper left", bbox_to_anchor=(1.15, 1))
    
    plt.tight_layout()
    if out_dir:
        fig.savefig(out_dir / f"heatmap_{metric}{fname_suffix}.png", dpi=300)
    plt.show()


# --- classification heatmaps ---
if len(df_cls) > 0:
    print("=== classification (all outputs, including gencode) ===")
    sub = df_cls[df_cls.subset == "all"].copy()
    sub["base_model"] = sub.output.apply(_get_base_model)
    sub = sub[~sub.base_model.isin(exclude_bases)]
    
    # average across acceptor/donor and samples
    agg = sub.groupby(["tissue", "output"]).agg(
        auprc=("auprc", "mean"),
        auroc=("auroc", "mean"),
        topk=("topk", "mean"),
    ).reset_index()
    
    # melt to long format for plotting function
    cls_long = agg.melt(id_vars=["tissue", "output"], var_name="metric", value_name="score")
    
    for m in ["auprc", "auroc", "topk"]:
        plot_heatmap_all_outputs(cls_long, m, tissue_list=_heatmap_tissues_cls, out_dir=fig2_sup)
else:
    print("no classification data")

# --- regression heatmaps (all subsets) - expression tissues only ---
subset_labels = {
    "ssu_valid": "all sites",
    "ssu_valid_nonzero": "SSU > 0",
    "ssu_shared": "shared sites",
    "ssu_shared_nonzero": "shared, SSU > 0",
}

if len(df_reg) > 0:
    for subset, subset_label in subset_labels.items():
        print(f"\n=== regression - {subset_label} (all outputs, expression only) ===")
        
        sub = df_reg[df_reg.subset == subset].copy()
        sub["base_model"] = sub.output.apply(_get_base_model)
        sub = sub[~sub.base_model.isin(exclude_bases)]
        
        if len(sub) == 0:
            print(f"no data for {subset}")
            continue
        
        # average across samples
        agg_cols = ["pearson", "spearman", "r2", "mse"]
        if "mae" in sub.columns:
            agg_cols.append("mae")
        
        agg = sub.groupby(["tissue", "output"]).agg(
            {col: "mean" for col in agg_cols}
        ).reset_index()
        
        # melt to long format
        reg_long = agg.melt(id_vars=["tissue", "output"], var_name="metric", value_name="score")
        
        title_suffix = f" ({subset_label})"
        fname_suffix = f"_{subset}"
        
        for m in agg_cols:
            plot_heatmap_all_outputs(reg_long, m, title_suffix=title_suffix, fname_suffix=fname_suffix, tissue_list=_heatmap_tissues, out_dir=fig2_sup)
else:
    print("no regression data")

## Supplemental: Binned Metrics + Sample Sizes

AUPRC and regression MAE stratified by SSU bin, plus sample size breakdowns.

In [ ]:
# binned AUPRC: ratio-preserving strategy
# line plot by SSU bin with sample counts on secondary axis

bin_labels = [f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)]

# explicit order matching MSE plot: spliceai last
cls_order = ["sphaec_ref", "pangolin_avg_p_splice", "splicetransformer", "spliceai"]
reg_order = ["sphaec_ref_ssu", "pangolin_avg_usage", "splicetransformer_avg_tissue"]
all_order = cls_order + reg_order


for tissue in tissues:
    sub = df_bin_ratio[(df_bin_ratio.tissue == tissue) & (df_bin_ratio.subset == "ssu_valid_nonzero")].copy()
    if len(sub) == 0:
        continue
    
    available = sub.output.unique().tolist()
    
    # filter to outputs that exist, maintaining order
    output_order = [o for o in all_order if o in available]
    if len(output_order) == 0:
        print(f"{tissue}: no matching outputs in {available}")
        continue
    
    print(f"outputs used ({tissue}): {[get_name(o) for o in output_order]}")
    
    sub = sub[sub.output.isin(output_order)]
    
    # average acceptor/donor
    agg = sub.groupby(["output", "bin"]).agg(
        auprc=("auprc", "mean"),
        auprc_std=("auprc", "std"),
        n_pos=("n_pos", "mean"),
        n_neg=("n_neg", "mean"),
    ).reset_index()
    
    # get counts for one model (same across models)
    counts = agg[agg.output == output_order[0]][["bin", "n_pos", "n_neg"]].sort_values("bin")
    
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # secondary axis for sample counts (subtle gray bars)
    ax2 = ax.twinx()
    bar_width = 0.5
    bars = ax2.bar(counts["bin"], counts["n_pos"], width=bar_width, color=GRAY_FAINT, 
                   alpha=0.4, zorder=1, label="n sites")
    ax2.set_ylabel("n sites", color=GRAY_MED)
    ax2.tick_params(axis="y", labelcolor=GRAY_MED)
    ax2.set_yscale("log")
    ax2.set_ylim(counts["n_pos"].min() * 0.5, counts["n_pos"].max() * 3)
    
    # main line plots (on top) in explicit order
    for out in output_order:
        out_df = agg[agg.output == out].sort_values("bin")
        _ls = "--" if out in reg_order else "-"
        ax.plot(out_df["bin"], out_df["auprc"], marker="o", label=get_name(out),
                color=get_color(out), lw=2, markersize=6, zorder=3, ls=_ls)
    
    ax.set_xticks(range(10))
    ax.set_xticklabels(bin_labels, rotation=45, ha="right")
    ax.set_xlabel("SSU bin")
    ax.set_ylabel("AUPRC")
    ax.set_ylim(0, 1)
    
    ax.tick_params(axis='both')                                                                                                 
    
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=4, frameon=False)
    
    ax.set_title(f"{tissue_display.get(tissue, tissue)}")
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)
    
    plt.tight_layout()
    plt.savefig(fig2_main / f"binned_auprc_{tissue}.png", dpi=300)
    plt.show()

# --- vertical stacked binned AUPRC (no legend, shared x-ticks) ---
def _match_tissue(desired, available):
    """match desired short name to actual tissue name via substring"""
    for t in available:
        if desired in t.lower():
            return t
    return None
_desired_v = ["haec", "lung", "brain", "testis", "blood"]
tissues_v_order = [m for d in _desired_v if (m := _match_tissue(d, tissues_fig1)) is not None]

nt_v = len(tissues_v_order)
fig, axes = plt.subplots(nt_v, 1, figsize=(6, 4 * nt_v), sharex=True)
for row, tissue in enumerate(tissues_v_order):
    ax = axes[row]
    sub_v = df_bin_ratio[(df_bin_ratio.tissue == tissue) & (df_bin_ratio.subset == "ssu_valid_nonzero")].copy()
    if len(sub_v) == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(tissue_display.get(tissue, tissue))
        continue

    avail_v = [o for o in all_order if o in sub_v.output.unique().tolist()]
    sub_v = sub_v[sub_v.output.isin(avail_v)]
    agg_v = sub_v.groupby(["output", "bin"]).agg(
        auprc=("auprc", "mean"),
        n_pos=("n_pos", "mean"),
    ).reset_index()

    counts_v = agg_v[agg_v.output == avail_v[0]][["bin", "n_pos"]].sort_values("bin")
    ax2 = ax.twinx()
    ax2.bar(counts_v["bin"], counts_v["n_pos"], width=0.5, color=GRAY_FAINT, alpha=0.4, zorder=1)
    ax2.set_ylabel("n sites" if row == nt_v // 2 else "", color=GRAY_MED)
    ax2.tick_params(axis="y", labelcolor=GRAY_MED)
    ax2.set_yscale("log")
    ax2.set_ylim(counts_v["n_pos"].min() * 0.5, counts_v["n_pos"].max() * 3)

    for out in avail_v:
        out_df = agg_v[agg_v.output == out].sort_values("bin")
        _ls = "--" if out in reg_order else "-"
        ax.plot(out_df["bin"], out_df["auprc"], marker="o",
                color=get_color(out), lw=2, markersize=6, zorder=3, ls=_ls)

    ax.set_ylim(0, 1)
    ax.set_ylabel("AUPRC")
    ax.set_title(tissue_display.get(tissue, tissue))
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

    if row == nt_v - 1:
        ax.set_xticks(range(10))
        ax.set_xticklabels(bin_labels, rotation=45, ha="right")
        ax.set_xlabel("SSU bin")
    else:
        ax.set_xticklabels([])
        ax.set_xlabel("")

plt.tight_layout()
plt.savefig(fig2_main / "binned_auprc_v.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved binned_auprc_v.png")


# --- horizontal stacked binned AUPRC (no legend, shared y-axis) ---
nt_h = len(tissues_v_order)
fig, axes = plt.subplots(1, nt_h, figsize=(6 * nt_h, 5), sharey=True)
for col, tissue in enumerate(tissues_v_order):
    ax = axes[col]
    sub_h = df_bin_ratio[(df_bin_ratio.tissue == tissue) & (df_bin_ratio.subset == "ssu_valid_nonzero")].copy()
    if len(sub_h) == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(tissue_display.get(tissue, tissue))
        continue

    avail_h = [o for o in all_order if o in sub_h.output.unique().tolist()]
    sub_h = sub_h[sub_h.output.isin(avail_h)]
    agg_h = sub_h.groupby(["output", "bin"]).agg(
        auprc=("auprc", "mean"),
        n_pos=("n_pos", "mean"),
    ).reset_index()

    counts_h = agg_h[agg_h.output == avail_h[0]][["bin", "n_pos"]].sort_values("bin")
    ax2 = ax.twinx()
    ax2.bar(counts_h["bin"], counts_h["n_pos"], width=0.5, color=GRAY_FAINT, alpha=0.4, zorder=1)
    ax2.tick_params(axis="y", labelcolor=GRAY_MED)
    ax2.set_yscale("log")
    ax2.set_ylim(counts_h["n_pos"].min() * 0.5, counts_h["n_pos"].max() * 3)
    if col == nt_h - 1:
        ax2.set_ylabel("n sites", color=GRAY_MED)
    else:
        ax2.set_yticklabels([])

    for out in avail_h:
        out_df = agg_h[agg_h.output == out].sort_values("bin")
        _ls = "--" if out in reg_order else "-"
        ax.plot(out_df["bin"], out_df["auprc"], marker="o",
                color=get_color(out), lw=2, markersize=6, zorder=3, ls=_ls)

    ax.set_ylim(0, 1)
    if col == 0:
        ax.set_ylabel("AUPRC")
    ax.set_title(tissue_display.get(tissue, tissue))
    ax.set_xticks(range(10))
    ax.set_xticklabels(bin_labels, rotation=45, ha="right")
    ax.set_xlabel("SSU bin")
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

plt.tight_layout()
plt.savefig(fig2_main / "binned_auprc_h.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved binned_auprc_h.png")

In [ ]:
# binned regression: MAE by SSU bin with sample counts
# solid = classification, dashed = regression (SSU)

bin_labels = [f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)]

# specify order here - will appear in legend with 2 columns
output_order_spec = [
    "sphaec_ref_cls",
    "sphaec_ref_ssu",
    "pangolin_avg_p_splice",
    "pangolin_avg_usage",
    "splicetransformer_cls",
    "splicetransformer_avg_tissue",
    "spliceai_cls",
]

# which outputs are regression (dashed lines, " - SSU" suffix)
reg_outputs = ["sphaec_ref_ssu", "pangolin_avg_usage", "splicetransformer_avg_tissue"]

for tissue in tissues:
    print(f"\n=== Processing {tissue} ===")
    sub = df_bin_reg[(df_bin_reg.tissue == tissue) & (df_bin_reg.subset == "ssu_valid_nonzero")].copy()
    if len(sub) == 0:
        print(f"  no data for {tissue}")
        continue
    
    available = sub.output.unique().tolist()
    output_order = [o for o in output_order_spec if o in available]
    
    if len(output_order) == 0:
        print(f"  no matching outputs in {available}")
        continue
    
    print(f"  using: {[get_name(o) for o in output_order]}")
    
    sub = sub[sub.output.isin(output_order)]
    metric_col = "mae" if "mae" in sub.columns else "mse"
    
    agg = sub.groupby(["output", "bin"]).agg(
        metric=(metric_col, "mean"),
        n=("n", "mean"),
    ).reset_index()
    
    counts = agg[agg.output == output_order[0]][["bin", "n"]].sort_values("bin")
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # secondary axis for sample counts (no gaps between bars)
    ax2 = ax.twinx()
    ax2.bar(counts["bin"], counts["n"], width=0.5, color=GRAY_FAINT, alpha=0.4, zorder=1)
    ax2.set_ylabel("n sites", color=GRAY_MED)
    ax2.tick_params(axis="y", labelcolor=GRAY_MED)
    ax2.set_yscale("log")
    ax2.set_ylim(counts["n"].min() * 0.5, counts["n"].max() * 3)
    
    # plot in specified order
    for out in output_order:
        out_df = agg[agg.output == out].sort_values("bin")
        is_reg = out in reg_outputs
        ls = "--" if is_reg else "-"
        label = get_name(out) + " - SSU" if is_reg else get_name(out)
        ax.plot(out_df["bin"], out_df["metric"], marker="o", label=label,
                color=get_color(out), lw=2, markersize=6, zorder=3, ls=ls)
    
    ax.set_xticks(range(10))
    ax.set_xticklabels(bin_labels, rotation=45, ha="right")
    ax.set_xlabel("SSU bin")
    ax.set_ylabel("Mean Squared Error")
    
    ax.tick_params(axis='both')
    
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.28), ncol=2, 
              frameon=False, handlelength=3, markerscale=1)
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)
    
    ax.set_title(f"{tissue_display.get(tissue, tissue)}")
    plt.tight_layout()
    plt.savefig(fig2_main / f"binned_regression_{tissue}.png", dpi=300)
    plt.show()

# --- vertical stacked binned regression (no legend, shared axes) ---
def _match_tissue(desired, available):
    """match desired short name to actual tissue name via substring"""
    for t in available:
        if desired in t.lower():
            return t
    return None
_desired_v = ["haec", "lung", "brain", "testis", "blood"]
tissues_v_order = [m for d in _desired_v if (m := _match_tissue(d, tissues_fig1)) is not None]

# first pass: shared y-axis limits
_metric_col = "mae" if "mae" in df_bin_reg.columns else "mse"
_y_vals = []
for t in tissues_v_order:
    sub_v = df_bin_reg[(df_bin_reg.tissue == t) & (df_bin_reg.subset == "ssu_valid_nonzero")]
    if len(sub_v) == 0:
        continue
    avail_v = [o for o in output_order_spec if o in sub_v.output.unique().tolist()]
    sub_v = sub_v[sub_v.output.isin(avail_v)]
    agg_v = sub_v.groupby(["output", "bin"]).agg(metric=(_metric_col, "mean")).reset_index()
    _y_vals.extend(agg_v["metric"].tolist())
_y_min_v = 0
_y_max_v = max(_y_vals) * 1.1 if _y_vals else 1.0

nt_v = len(tissues_v_order)
fig, axes = plt.subplots(nt_v, 1, figsize=(8, 4 * nt_v), sharex=True)
for row, tissue in enumerate(tissues_v_order):
    ax = axes[row]
    sub_v = df_bin_reg[(df_bin_reg.tissue == tissue) & (df_bin_reg.subset == "ssu_valid_nonzero")].copy()
    if len(sub_v) == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(tissue_display.get(tissue, tissue))
        continue

    avail_v = [o for o in output_order_spec if o in sub_v.output.unique().tolist()]
    sub_v = sub_v[sub_v.output.isin(avail_v)]
    agg_v = sub_v.groupby(["output", "bin"]).agg(
        metric=(_metric_col, "mean"),
        n=("n", "mean"),
    ).reset_index()

    # sample count bars
    counts_v = agg_v[agg_v.output == avail_v[0]][["bin", "n"]].sort_values("bin")
    ax2 = ax.twinx()
    ax2.bar(counts_v["bin"], counts_v["n"], width=0.5, color=GRAY_FAINT, alpha=0.4, zorder=1)
    ax2.set_ylabel("n sites" if row == nt_v // 2 else "", color=GRAY_MED)
    ax2.tick_params(axis="y", labelcolor=GRAY_MED)
    ax2.set_yscale("log")
    ax2.set_ylim(counts_v["n"].min() * 0.5, counts_v["n"].max() * 3)

    for out in avail_v:
        out_df = agg_v[agg_v.output == out].sort_values("bin")
        is_reg = out in reg_outputs
        ls = "--" if is_reg else "-"
        ax.plot(out_df["bin"], out_df["metric"], marker="o",
                color=get_color(out), lw=2, markersize=6, zorder=3, ls=ls)

    ax.set_ylabel("Mean Squared Error")
    ax.set_title(tissue_display.get(tissue, tissue))
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

    if row == nt_v - 1:
        ax.set_xticks(range(10))
        ax.set_xticklabels(bin_labels, rotation=45, ha="right")
        ax.set_xlabel("SSU bin")
    else:
        ax.set_xticklabels([])
        ax.set_xlabel("")

plt.tight_layout()
plt.savefig(fig2_main / "binned_regression_v.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved binned_regression_v.png")


# --- horizontal stacked binned regression (no legend, shared y-axis) ---
nt_h = len(tissues_v_order)
fig, axes = plt.subplots(1, nt_h, figsize=(6 * nt_h, 5), sharey=True)
for col, tissue in enumerate(tissues_v_order):
    ax = axes[col]
    sub_h = df_bin_reg[(df_bin_reg.tissue == tissue) & (df_bin_reg.subset == "ssu_valid_nonzero")].copy()
    if len(sub_h) == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(tissue_display.get(tissue, tissue))
        continue

    avail_h = [o for o in output_order_spec if o in sub_h.output.unique().tolist()]
    sub_h = sub_h[sub_h.output.isin(avail_h)]
    agg_h = sub_h.groupby(["output", "bin"]).agg(
        metric=(_metric_col, "mean"),
        n=("n", "mean"),
    ).reset_index()

    counts_h = agg_h[agg_h.output == avail_h[0]][["bin", "n"]].sort_values("bin")
    ax2 = ax.twinx()
    ax2.bar(counts_h["bin"], counts_h["n"], width=0.5, color=GRAY_FAINT, alpha=0.4, zorder=1)
    ax2.tick_params(axis="y", labelcolor=GRAY_MED)
    ax2.set_yscale("log")
    ax2.set_ylim(counts_h["n"].min() * 0.5, counts_h["n"].max() * 3)
    if col == nt_h - 1:
        ax2.set_ylabel("n sites", color=GRAY_MED)
    else:
        ax2.set_yticklabels([])

    for out in avail_h:
        out_df = agg_h[agg_h.output == out].sort_values("bin")
        is_reg = out in reg_outputs
        ls = "--" if is_reg else "-"
        ax.plot(out_df["bin"], out_df["metric"], marker="o",
                color=get_color(out), lw=2, markersize=6, zorder=3, ls=ls)

    if col == 0:
        ax.set_ylabel("Mean Squared Error")
    ax.set_title(tissue_display.get(tissue, tissue))
    ax.set_xticks(range(10))
    ax.set_xticklabels(bin_labels, rotation=45, ha="right")
    ax.set_xlabel("SSU bin")
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

plt.tight_layout()
plt.savefig(fig2_main / "binned_regression_h.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved binned_regression_h.png")

In [ ]:
# sample sizes by tissue

_cls_labels = {"all": "all positions", **subset_labels}

# --- classification: n_pos by tissue, per subset ---
if len(df_cls) > 0:
    cls_subsets = [s for s in ["all", "ssu_valid", "ssu_valid_nonzero", "ssu_shared", "ssu_shared_nonzero"]
                   if s in df_cls.subset.unique()]

    for subset in cls_subsets:
        n_df = df_cls[df_cls.subset == subset].groupby(["tissue", "sample_id", "target"]).agg(
            n_pos=("n_pos", "first"),
            n_neg=("n_neg", "first"),
        ).reset_index()

        cls_tissues = [t for t in all_tissues if t in n_df.tissue.unique()]
        n_df = n_df[n_df.tissue.isin(cls_tissues)]

        if len(n_df) == 0:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        for ax, target in zip(axes, ["acceptor", "donor"]):
            sub = n_df[n_df.target == target]

            sns.stripplot(data=sub, x="tissue", y="n_pos", hue="tissue", order=cls_tissues,
                          palette={t: tissue_colors.get(t, "#999999") for t in cls_tissues},
                          size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)

            for i, tissue in enumerate(cls_tissues):
                vals = sub[sub.tissue == tissue].n_pos
                if len(vals) > 0:
                    ax.hlines(vals.mean(), i - 0.3, i + 0.3, color=tissue_colors.get(tissue, "#999999"), lw=2.5)

            n_expr = len([t for t in tissues if t in cls_tissues])
            if n_expr < len(cls_tissues):
                ax.axvline(x=n_expr - 0.5, color="gray", lw=1, ls="--", alpha=0.5)

            ax.set_xlabel("")
            ax.set_ylabel("n positives")
            ax.set_xticks(range(len(cls_tissues)))
            ax.set_xticklabels([tissue_display.get(t, t) for t in cls_tissues], rotation=45, ha="right")
            ax.set_title(f"{target.capitalize()} sites")

        fig.suptitle(f"Classification — {_cls_labels.get(subset, subset)}", y=1.02)
        plt.tight_layout()
        plt.savefig(fig2_sup / f"sample_sizes_cls_{subset}.png", dpi=300, bbox_inches="tight")
        plt.show()

        # summary table
        summary = n_df.groupby(["tissue", "target"]).agg(
            n_pos_mean=("n_pos", "mean"),
            n_pos_std=("n_pos", "std"),
            n_neg_mean=("n_neg", "mean"),
            n_samples=("sample_id", "nunique"),
        ).reset_index()

        def fmt_n(row):
            if pd.isna(row.n_pos_std) or row.n_pos_std == 0:
                return f"{row.n_pos_mean:,.0f}"
            return f"{row.n_pos_mean:,.0f} +/- {row.n_pos_std:,.0f}"

        summary["n_pos"] = summary.apply(fmt_n, axis=1)
        summary["n_neg"] = summary.n_neg_mean.apply(lambda x: f"{x:,.0f}")
        summary["n_samples"] = summary.n_samples.astype(int)
        summary["tissue_order"] = summary.tissue.apply(lambda t: all_tissues.index(t) if t in all_tissues else 999)
        summary = summary.sort_values(["tissue_order", "target"]).drop(columns="tissue_order")
        print(f"\nClassification — {_cls_labels.get(subset, subset)}:")
        print(summary[["tissue", "target", "n_pos", "n_neg", "n_samples"]].to_string(index=False))
else:
    print("no classification data")

# --- regression: n splice sites per subset, split by acceptor/donor ---
if len(df_reg) > 0:
    reg_tissues = [t for t in tissues if t in df_reg.tissue.unique()]

    for subset, subset_label in subset_labels.items():
        sub_reg = df_reg[df_reg.subset == subset]
        sub_reg = sub_reg[sub_reg.tissue.isin(reg_tissues)]
        if len(sub_reg) == 0:
            print(f"no data for {subset}")
            continue

        # get acceptor/donor split from classification data
        sub_cls = df_cls[(df_cls.subset == subset)] if len(df_cls) > 0 else pd.DataFrame()
        sub_cls = sub_cls[sub_cls.tissue.isin(reg_tissues)]

        if len(sub_cls) == 0:
            print(f"WARNING: no classification data for subset={subset}, cannot split by acceptor/donor")
            continue

        n_df_reg = sub_cls.groupby(["tissue", "sample_id", "target"]).agg(
            n_pos=("n_pos", "first"),
        ).reset_index()

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for ax, target in zip(axes, ["acceptor", "donor"]):
            tsub = n_df_reg[n_df_reg.target == target]

            sns.stripplot(data=tsub, x="tissue", y="n_pos", hue="tissue", order=reg_tissues,
                          palette={t: tissue_colors.get(t, "#999999") for t in reg_tissues},
                          size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)

            for i, tissue in enumerate(reg_tissues):
                vals = tsub[tsub.tissue == tissue].n_pos
                if len(vals) > 0:
                    ax.hlines(vals.mean(), i - 0.3, i + 0.3, color=tissue_colors.get(tissue, "#999999"), lw=2.5)

            ax.set_xlabel("")
            ax.set_ylabel("n splice sites")
            ax.set_xticks(range(len(reg_tissues)))
            ax.set_xticklabels([tissue_display.get(t, t) for t in reg_tissues], rotation=45, ha="right")
            ax.set_title(f"{target.capitalize()} sites")

        fig.suptitle(f"Regression — {subset_label}", y=1.02)
        plt.tight_layout()
        plt.savefig(fig2_sup / f"sample_sizes_reg_{subset}.png", dpi=300, bbox_inches="tight")
        plt.show()

        # summary split by acceptor/donor
        n_df_summary = sub_cls.groupby(["tissue", "target"]).agg(
            n_mean=("n_pos", "mean"),
            n_std=("n_pos", "std"),
            n_samples=("sample_id", "nunique"),
        ).reset_index()
        n_df_summary["tissue_order"] = n_df_summary.tissue.apply(lambda t: reg_tissues.index(t) if t in reg_tissues else 999)
        n_df_summary = n_df_summary.sort_values(["tissue_order", "target"]).drop(columns="tissue_order")
        print(f"\n{subset_label}:")
        for _, row in n_df_summary.iterrows():
            if pd.isna(row.n_std) or row.n_std == 0:
                print(f"  {row.tissue} ({row.target}): {row.n_mean:,.0f} sites ({int(row.n_samples)} samples)")
            else:
                print(f"  {row.tissue} ({row.target}): {row.n_mean:,.0f} +/- {row.n_std:,.0f} sites ({int(row.n_samples)} samples)")
else:
    print("no regression data")

## Alternative Compositions

Swarm/bar plots, pangolin test set bars, combined multi-panel figure, and compact all-tissues layout.

In [ ]:
# separate swarm and bar plots

def _match_tissue(desired, available):
    """match desired short name to actual tissue name via substring"""
    for t in available:
        if desired in t.lower():
            return t
    return None
_desired_v = ["haec", "lung", "brain", "testis", "blood"]
tissues_v_order = [m for d in _desired_v if (m := _match_tissue(d, tissues_fig1)) is not None]

exclude_bases = {"sphaec_var", "sphaec_avg"}

# r² swarm plots config (use global r2_outputs)
r2_display = {
    "sphaec_ref_ssu": "SPLAIRE",
    "spliceai_cls": "SpliceAI",
    "pangolin_avg_usage": "Pangolin",
    "splicetransformer_avg_tissue": "SpliceTransformer",
}

# bar plots config (classification outputs)
bar_outputs = ["sphaec_ref", "spliceai", "pangolin_avg_p_splice", "splicetransformer"]
bar_outputs_with_gencode = ["gencode", "sphaec_ref", "spliceai", "pangolin_avg_p_splice", "splicetransformer"]
bar_display = {
    "gencode": "GENCODE",
    "mane_select": "MANE",
    "sphaec_ref": "SPLAIRE",
    "spliceai": "SpliceAI",
    "pangolin_avg_p_splice": "Pangolin",
    "splicetransformer": "SpliceTransformer",
}

# tissue lists for different plot types
tissues_with_gencode = tissues_fig1 + ["gencode"]
tissues_with_mane = tissues_fig1 + ["mane_select"]
tissues_with_both = tissues_fig1 + ["gencode", "mane_select"]

# mapping from r2 outputs to bar outputs for consistent ordering
r2_to_bar = {
    "sphaec_ref_ssu": "sphaec_ref",
    "spliceai_cls": "spliceai",
    "pangolin_avg_usage": "pangolin_avg_p_splice",
    "splicetransformer_avg_tissue": "splicetransformer",
}

# regression subsets to plot
reg_subsets = [
    ("ssu_valid", "all sites", "all"),
    ("ssu_valid_nonzero", "SSU > 0", "nonzero"),
    ("ssu_shared", "shared sites", "shared"),
    ("ssu_shared_nonzero", "shared, SSU > 0", "shared_nonzero"),
]

# --- r² swarm plots (all subsets) ---
for subset_name, subset_label, suffix in reg_subsets:
    sub = df_reg[df_reg.subset == subset_name].copy()
    sub["base_model"] = sub.output.apply(_get_base_model)
    sub = sub[~sub.base_model.isin(exclude_bases)]
    
    available = [o for o in r2_outputs if o in sub.output.unique()]
    sub = sub[sub.output.isin(available)]
    output_order = list(sub.groupby("output").r2.mean().sort_values(ascending=False).index)
    print(f"r2 swarm ({subset_label}):", output_order)
    
    fig, axes = plt.subplots(1, len(tissues_fig1), figsize=(3 * len(tissues_fig1), 4), sharey=True)
    for col, tissue in enumerate(tissues_fig1):
        ax = axes[col]
        tdf = sub[sub.tissue == tissue]
    
        sns.stripplot(data=tdf, x="output", y="r2", hue="output", order=output_order,
                      palette={o: get_color(o) for o in output_order},
                      size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)
    
        for i, out in enumerate(output_order):
            vals = tdf[tdf.output == out].r2
            if len(vals) == 0:
                continue
            c = get_color(out)
            ax.hlines(vals.mean(), i - 0.3, i + 0.3, color=c, lw=2)
            ax.hlines(vals.median(), i - 0.3, i + 0.3, color=c, lw=1.5, ls="--")
    
        ax.set_xlabel("")
        ax.set_ylabel(r"$R^2$" if col == 0 else "")
        ax.set_title(tissue_display.get(tissue, tissue))
        ax.set_xticks(range(len(output_order)))
        ax.set_xticklabels([r2_display.get(o, o) for o in output_order], rotation=45, ha="right")
    
    legend_elements = [Line2D([0], [0], color="gray", lw=2, label="mean"),
                       Line2D([0], [0], color="gray", lw=1.5, ls="--", label="median")]
    fig.suptitle(f"R² ({subset_label})", y=1.05)
    fig.legend(handles=legend_elements, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout()
    plt.savefig(fig2_main / f"r2_swarm_{suffix}.png", dpi=300, bbox_inches="tight")
    plt.show()

    # vertical version
    fig, axes = plt.subplots(len(tissues_v_order), 1, figsize=(4, 4 * len(tissues_v_order)))
    for row, tissue in enumerate(tissues_v_order):
        ax = axes[row]
        tdf = sub[sub.tissue == tissue]
        sns.stripplot(data=tdf, x="output", y="r2", hue="output", order=output_order,
                      palette={o: get_color(o) for o in output_order},
                      size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)
        for i, out in enumerate(output_order):
            vals = tdf[tdf.output == out].r2
            if len(vals) == 0:
                continue
            c = get_color(out)
            ax.hlines(vals.mean(), i - 0.3, i + 0.3, color=c, lw=2)
            ax.hlines(vals.median(), i - 0.3, i + 0.3, color=c, lw=1.5, ls="--")
        ax.set_xlabel("")
        ax.set_ylabel(r"$R^2$")
        ax.set_title(tissue_display.get(tissue, tissue))
        ax.set_xticks(range(len(output_order)))
        if row == len(tissues_v_order) - 1:
            ax.set_xticklabels([r2_display.get(o, o) for o in output_order], rotation=45, ha="right")
        else:
            ax.set_xticklabels([])
    legend_elements_v = [Line2D([0], [0], color="gray", lw=2, label="mean"),
                         Line2D([0], [0], color="gray", lw=1.5, ls="--", label="median")]
    fig.suptitle(f"R² ({subset_label})", y=1.02)
    fig.legend(handles=legend_elements_v, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 1.00))
    plt.tight_layout()
    plt.savefig(fig2_main / f"r2_swarm_{suffix}_v.png", dpi=300, bbox_inches="tight")
    plt.show()

    # individual per-tissue swarm plots
    y_min = sub.r2.min() - 0.05
    y_max = sub.r2.max() + 0.05
    for tissue in tissues_fig1:
        tdf = sub[sub.tissue == tissue]
        if tdf.empty:
            continue
        fig_ind, ax_ind = plt.subplots(1, 1, figsize=(3, 4))
        sns.stripplot(data=tdf, x="output", y="r2", hue="output", order=output_order,
                      palette={o: get_color(o) for o in output_order},
                      size=5, jitter=0.2, alpha=0.7, ax=ax_ind, legend=False)
        for i, out in enumerate(output_order):
            vals = tdf[tdf.output == out].r2
            if len(vals) == 0:
                continue
            c = get_color(out)
            ax_ind.hlines(vals.mean(), i - 0.3, i + 0.3, color=c, lw=2)
            ax_ind.hlines(vals.median(), i - 0.3, i + 0.3, color=c, lw=1.5, ls="--")
        ax_ind.set_xlabel("")
        ax_ind.set_ylabel(r"$R^2$")
        ax_ind.set_title(tissue_display.get(tissue, tissue))
        ax_ind.set_xticks(range(len(output_order)))
        ax_ind.set_xticklabels([r2_display.get(o, o) for o in output_order], rotation=45, ha="right")
        ax_ind.set_ylim(y_min, y_max)
        plt.tight_layout()
        plt.savefig(fig2_main / f"r2_swarm_{suffix}_{tissue}.png", dpi=300, bbox_inches="tight")
        plt.show()


# --- pearson swarm plots (all subsets) ---
for subset_name, subset_label, suffix in reg_subsets:
    sub = df_reg[df_reg.subset == subset_name].copy()
    sub["base_model"] = sub.output.apply(_get_base_model)
    sub = sub[~sub.base_model.isin(exclude_bases)]
    
    available = [o for o in r2_outputs if o in sub.output.unique()]
    sub = sub[sub.output.isin(available)]
    output_order_p = list(sub.groupby("output").pearson.mean().sort_values(ascending=False).index)
    print(f"pearson swarm ({subset_label}):", output_order_p)
    
    fig, axes = plt.subplots(1, len(tissues_fig1), figsize=(3 * len(tissues_fig1), 4), sharey=True)
    for col, tissue in enumerate(tissues_fig1):
        ax = axes[col]
        tdf = sub[sub.tissue == tissue]
    
        sns.stripplot(data=tdf, x="output", y="pearson", hue="output", order=output_order_p,
                      palette={o: get_color(o) for o in output_order_p},
                      size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)
    
        for i, out in enumerate(output_order_p):
            vals = tdf[tdf.output == out].pearson
            if len(vals) == 0:
                continue
            c = get_color(out)
            ax.hlines(vals.mean(), i - 0.3, i + 0.3, color=c, lw=2)
            ax.hlines(vals.median(), i - 0.3, i + 0.3, color=c, lw=1.5, ls="--")
    
        ax.set_xlabel("")
        ax.set_ylabel("Pearson r" if col == 0 else "")
        ax.set_title(tissue_display.get(tissue, tissue))
        ax.set_xticks(range(len(output_order_p)))
        ax.set_xticklabels([r2_display.get(o, o) for o in output_order_p], rotation=45, ha="right")
    
    legend_elements = [Line2D([0], [0], color="gray", lw=2, label="mean"),
                       Line2D([0], [0], color="gray", lw=1.5, ls="--", label="median")]
    fig.suptitle(f"Pearson r ({subset_label})", y=1.05)
    fig.legend(handles=legend_elements, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout()
    plt.savefig(fig2_main / f"pearson_swarm_{suffix}.png", dpi=300, bbox_inches="tight")
    plt.show()

    # vertical version
    fig, axes = plt.subplots(len(tissues_v_order), 1, figsize=(4, 4 * len(tissues_v_order)))
    for row, tissue in enumerate(tissues_v_order):
        ax = axes[row]
        tdf = sub[sub.tissue == tissue]
        sns.stripplot(data=tdf, x="output", y="pearson", hue="output", order=output_order_p,
                      palette={o: get_color(o) for o in output_order_p},
                      size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)
        for i, out in enumerate(output_order_p):
            vals = tdf[tdf.output == out].pearson
            if len(vals) == 0:
                continue
            c = get_color(out)
            ax.hlines(vals.mean(), i - 0.3, i + 0.3, color=c, lw=2)
            ax.hlines(vals.median(), i - 0.3, i + 0.3, color=c, lw=1.5, ls="--")
        ax.set_xlabel("")
        ax.set_ylabel("Pearson r")
        ax.set_title(tissue_display.get(tissue, tissue))
        ax.set_xticks(range(len(output_order_p)))
        if row == len(tissues_v_order) - 1:
            ax.set_xticklabels([r2_display.get(o, o) for o in output_order_p], rotation=45, ha="right")
        else:
            ax.set_xticklabels([])
    legend_elements_v = [Line2D([0], [0], color="gray", lw=2, label="mean"),
                         Line2D([0], [0], color="gray", lw=1.5, ls="--", label="median")]
    fig.suptitle(f"Pearson r ({subset_label})", y=1.02)
    fig.legend(handles=legend_elements_v, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 1.00))
    plt.tight_layout()
    plt.savefig(fig2_main / f"pearson_swarm_{suffix}_v.png", dpi=300, bbox_inches="tight")
    plt.show()

    # individual per-tissue swarm plots
    y_min = sub.pearson.min() - 0.05
    y_max = sub.pearson.max() + 0.05
    for tissue in tissues_fig1:
        tdf = sub[sub.tissue == tissue]
        if tdf.empty:
            continue
        fig_ind, ax_ind = plt.subplots(1, 1, figsize=(3, 4))
        sns.stripplot(data=tdf, x="output", y="pearson", hue="output", order=output_order_p,
                      palette={o: get_color(o) for o in output_order_p},
                      size=5, jitter=0.2, alpha=0.7, ax=ax_ind, legend=False)
        for i, out in enumerate(output_order_p):
            vals = tdf[tdf.output == out].pearson
            if len(vals) == 0:
                continue
            c = get_color(out)
            ax_ind.hlines(vals.mean(), i - 0.3, i + 0.3, color=c, lw=2)
            ax_ind.hlines(vals.median(), i - 0.3, i + 0.3, color=c, lw=1.5, ls="--")
        ax_ind.set_xlabel("")
        ax_ind.set_ylabel("Pearson r")
        ax_ind.set_title(tissue_display.get(tissue, tissue))
        ax_ind.set_xticks(range(len(output_order_p)))
        ax_ind.set_xticklabels([r2_display.get(o, o) for o in output_order_p], rotation=45, ha="right")
        ax_ind.set_ylim(y_min, y_max)
        plt.tight_layout()
        plt.savefig(fig2_main / f"pearson_swarm_{suffix}_{tissue}.png", dpi=300, bbox_inches="tight")
        plt.show()

# get output order from the last swarm plot for bar plots
output_order_bar = [r2_to_bar[o] for o in output_order if r2_to_bar.get(o) in bar_outputs]


# --- helper function to create bar plots ---
def make_bar_plots(df, metric_col, ylabel, output_list, display_dict, filename_suffix, tissue_list=None, vertical=False):
    """create bar plots for a metric across tissues"""
    if tissue_list is None:
        tissue_list = tissues_fig1
    bar_df = df.copy()
    bar_df = bar_df.groupby(["tissue", "sample_id", "output"]).agg(val=(metric_col, "mean")).reset_index()
    bar_df["base_model"] = bar_df.output.apply(_get_base_model)
    bar_df = bar_df[~bar_df.base_model.isin(exclude_bases)]
    
    available = [o for o in output_list if o in bar_df.output.unique()]
    bar_df = bar_df[bar_df.output.isin(available)]
    
    order = [o for o in output_order_bar if o in available]
    
    print(f"{ylabel} bars ({filename_suffix}):", order)
    
    nt = len(tissue_list)
    if vertical:
        fig, axes = plt.subplots(nt, 1, figsize=(4, 4 * nt))
    else:
        fig, axes = plt.subplots(1, nt, figsize=(3 * nt, 4), sharey=True)
    if nt == 1:
        axes = [axes]
    _bar_fs = plt.rcParams["font.size"] * 1.25 if vertical else None
    for col, tissue in enumerate(tissue_list):
        ax = axes[col]
        tdf = bar_df[bar_df.tissue == tissue]
        
        agg = tdf.groupby("output").agg(mean=("val", "mean")).reset_index()
        agg = agg.set_index("output").reindex(order).reset_index()
        
        x = np.arange(len(order))
        colors = [get_color(o) for o in order]
        
        bars = ax.bar(x, agg["mean"], color=colors, edgecolor="black", linewidth=0.5, alpha=0.8)
        
        for bar, val in zip(bars, agg["mean"]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                    f"{val:.2f}", ha="center", va="center", fontweight="bold", rotation=90, color="white", fontsize=_bar_fs)
        
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        ax.set_title(tissue_display.get(tissue, tissue))
        ax.set_xticks(x)
        if not vertical or col == nt - 1:
            ax.set_xticklabels([display_dict.get(o, o) for o in order], rotation=45, ha="right")
        else:
            ax.set_xticklabels([])
        ax.set_ylim(0, 1.1)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(fig2_main / f"{filename_suffix}.png", dpi=300)
    plt.show()

# --- top-k bar plots (without gencode/mane) ---
make_bar_plots(df_cls[df_cls.subset == "all"], "topk", "Top-k", bar_outputs, bar_display, "topk_bars", tissue_list=tissues_fig1)

# --- top-k bar plots (with gencode) ---
make_bar_plots(df_cls[df_cls.subset == "all"], "topk", "Top-k", bar_outputs, bar_display, "topk_bars_with_gencode", tissue_list=tissues_with_gencode)

# --- top-k bar plots (with mane) ---
make_bar_plots(df_cls[df_cls.subset == "all"], "topk", "Top-k", bar_outputs, bar_display, "topk_bars_with_mane", tissue_list=tissues_with_mane)

# --- top-k bar plots (with both gencode and mane) ---
make_bar_plots(df_cls[df_cls.subset == "all"], "topk", "Top-k", bar_outputs, bar_display, "topk_bars_with_both", tissue_list=tissues_with_both)

# --- AUPRC bar plots (without gencode/mane) ---
make_bar_plots(df_cls[df_cls.subset == "all"], "auprc", "AUPRC", bar_outputs, bar_display, "auprc_bars", tissue_list=tissues_fig1)

# --- AUPRC bar plots (with gencode) ---
make_bar_plots(df_cls[df_cls.subset == "all"], "auprc", "AUPRC", bar_outputs, bar_display, "auprc_bars_with_gencode", tissue_list=tissues_with_gencode)

# --- AUPRC bar plots (with mane) ---
make_bar_plots(df_cls[df_cls.subset == "all"], "auprc", "AUPRC", bar_outputs, bar_display, "auprc_bars_with_mane", tissue_list=tissues_with_mane)

# --- AUPRC bar plots (with both gencode and mane) ---
make_bar_plots(df_cls[df_cls.subset == "all"], "auprc", "AUPRC", bar_outputs, bar_display, "auprc_bars_with_both", tissue_list=tissues_with_both)





# --- vertical versions of combined bar plots ---
make_bar_plots(df_cls[df_cls.subset == "all"], "topk", "Top-k", bar_outputs, bar_display, "topk_bars_v", tissue_list=tissues_v_order, vertical=True)
make_bar_plots(df_cls[df_cls.subset == "all"], "topk", "Top-k", bar_outputs, bar_display, "topk_bars_with_gencode_v", tissue_list=tissues_with_gencode, vertical=True)
make_bar_plots(df_cls[df_cls.subset == "all"], "topk", "Top-k", bar_outputs, bar_display, "topk_bars_with_mane_v", tissue_list=tissues_with_mane, vertical=True)
make_bar_plots(df_cls[df_cls.subset == "all"], "auprc", "AUPRC", bar_outputs, bar_display, "auprc_bars_v", tissue_list=tissues_v_order, vertical=True)
make_bar_plots(df_cls[df_cls.subset == "all"], "auprc", "AUPRC", bar_outputs, bar_display, "auprc_bars_with_gencode_v", tissue_list=tissues_with_gencode, vertical=True)
make_bar_plots(df_cls[df_cls.subset == "all"], "auprc", "AUPRC", bar_outputs, bar_display, "auprc_bars_with_mane_v", tissue_list=tissues_with_mane, vertical=True)

# --- individual bar plots per tissue per metric ---
all_tissues_individual = tissues_fig1 + ["gencode", "mane_select"]
individual_metrics = [("topk", "Top-k"), ("auprc", "AUPRC")]

for metric_col, ylabel in individual_metrics:
    bar_df = df_cls[df_cls.subset == "all"].copy()
    bar_df = bar_df.groupby(["tissue", "sample_id", "output"]).agg(val=(metric_col, "mean")).reset_index()
    bar_df["base_model"] = bar_df.output.apply(_get_base_model)
    bar_df = bar_df[~bar_df.base_model.isin(exclude_bases)]
    available = [o for o in bar_outputs if o in bar_df.output.unique()]
    bar_df = bar_df[bar_df.output.isin(available)]
    order = [o for o in output_order_bar if o in available]

    for tissue in all_tissues_individual:
        tdf = bar_df[bar_df.tissue == tissue]
        if tdf.empty:
            print(f"  {ylabel} {tissue}: no data, skipping")
            continue

        agg = tdf.groupby("output").agg(mean=("val", "mean")).reset_index()
        agg = agg.set_index("output").reindex(order).reset_index()

        fig, ax = plt.subplots(1, 1, figsize=(3, 4))
        x = np.arange(len(order))
        colors_bar = [get_color(o) for o in order]
        bars = ax.bar(x, agg["mean"], color=colors_bar, edgecolor="black", linewidth=0.5, alpha=0.8)
        for bar, val in zip(bars, agg["mean"]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                    f"{val:.2f}", ha="center", va="center", fontweight="bold", rotation=90, color="white")
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        ax.set_title(tissue_display.get(tissue, tissue))
        ax.set_xticks(x)
        ax.set_xticklabels([bar_display.get(o, o) for o in order], rotation=45, ha="right")
        ax.set_ylim(0, 1.1)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(fig2_main / f"{metric_col}_{tissue}.png", dpi=300, bbox_inches="tight")
        plt.show()

print(f"saved individual bar plots to {fig2_main}")

In [ ]:
# pangolin test set: bar plots by pangolin tissue
# same outputs as GTEx swarm+bar chunk above

# --- load pangolin test metrics ---
pang_h5 = base / "pangolin_test" / "metrics" / "pangolin_test_chr1357.h5"

pang_cls_rows = []
pang_reg_rows = []

with h5py.File(pang_h5, "r") as f:
    # per-tissue classification
    tissue_cls_grp = f.get("overall/classification_tissue")
    if tissue_cls_grp:
        for tissue in tissue_cls_grp.keys():
            for output in tissue_cls_grp[tissue].keys():
                m = tissue_cls_grp[tissue][output]
                pang_cls_rows.append({
                    "tissue": tissue, "output": output,
                    "auprc": float(m["auprc"][()]),
                    "auroc": float(m["auroc"][()]),
                    "topk": float(m["topk"][()]),
                    "n_pos": int(m["n_pos"][()]),
                    "n_neg": int(m["n_neg"][()]),
                })

    # per-tissue regression
    tissue_reg_grp = f.get("overall/regression_tissue")
    if tissue_reg_grp:
        for tissue in tissue_reg_grp.keys():
            for subset in tissue_reg_grp[tissue].keys():
                for output in tissue_reg_grp[tissue][subset].keys():
                    m = tissue_reg_grp[tissue][subset][output]
                    row = {
                        "tissue": tissue, "subset": subset, "output": output,
                        "pearson": float(m["pearson"][()]),
                        "spearman": float(m["spearman"][()]),
                        "r2": float(m["r2"][()]),
                        "mse": float(m["mse"][()]),
                        "n": int(m["n"][()]),
                    }
                    if "mae" in m:
                        row["mae"] = float(m["mae"][()])
                    pang_reg_rows.append(row)

df_pang_cls = pd.DataFrame(pang_cls_rows)
df_pang_reg = pd.DataFrame(pang_reg_rows)

pang_tissues = ["heart", "liver", "brain", "testis"]
pang_tissue_display = {"heart": "Heart", "liver": "Liver", "brain": "Brain", "testis": "Testis"}

# same outputs as GTEx chunk
# f1 = topk when k = n_pos
df_pang_cls.loc[:, "f1"] = df_pang_cls["topk"]

pang_cls_order = [o for o in bar_outputs if o in df_pang_cls.output.unique()]
pang_reg_order = [o for o in r2_outputs if o in df_pang_reg.output.unique()]

print(f"loaded {pang_h5.name}")
print(f"  classification: {df_pang_cls.shape}, tissues: {sorted(df_pang_cls.tissue.unique())}")
print(f"  regression: {df_pang_reg.shape}, tissues: {sorted(df_pang_reg.tissue.unique())}")
print(f"  cls outputs: {pang_cls_order}")
print(f"  reg outputs: {pang_reg_order}")

# --- bar plot helper ---

def make_pang_bar(df, metric, ylabel, output_order, display_dict, tissues, suffix, vertical=False):
    """bar plots for pangolin test (one value per tissue x output)"""
    sub = df[df.tissue.isin(tissues) & df.output.isin(output_order)].copy()
    nt = len(tissues)
    if vertical:
        fig, axes = plt.subplots(nt, 1, figsize=(4, 4 * nt))
    else:
        fig, axes = plt.subplots(1, nt, figsize=(3 * nt, 4), sharey=True)
    if nt == 1:
        axes = [axes]
    for col, tissue in enumerate(tissues):
        ax = axes[col]
        tdf = sub[sub.tissue == tissue].set_index("output").reindex(output_order)
        x = np.arange(len(output_order))
        colors = [get_color(o) for o in output_order]
        vals = tdf[metric].values

        bars = ax.bar(x, vals, color=colors, edgecolor="black", linewidth=0.5, alpha=0.8)
        for bar, v in zip(bars, vals):
            if np.isfinite(v):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                        f"{v:.2f}", ha="center", va="center", fontweight="bold",
                        rotation=90, color="white", fontsize=ANNOT_SIZE)

        ax.set_xlabel("")
        ax.set_ylabel(ylabel if col == 0 or vertical else "")
        ax.set_title(pang_tissue_display.get(tissue, tissue))
        ax.set_xticks(x)
        if not vertical or col == nt - 1:
            ax.set_xticklabels([display_dict.get(o, o) for o in output_order],
                               rotation=45, ha="right")
        else:
            ax.set_xticklabels([])
        if metric in {"auprc", "topk", "f1"}:
            ax.set_ylim(0, 1.1)

    plt.tight_layout()
    plt.savefig(fig2_main / f"pang_{suffix}.png", dpi=300, bbox_inches="tight")
    plt.show()

# --- classification bar plots (topk, auprc) ---
print("\n=== pangolin test classification ===")
make_pang_bar(df_pang_cls, "topk", "Top-k", pang_cls_order, bar_display, pang_tissues, "topk_bars")
make_pang_bar(df_pang_cls, "auprc", "AUPRC", pang_cls_order, bar_display, pang_tissues, "auprc_bars")

# vertical
make_pang_bar(df_pang_cls, "topk", "Top-k", pang_cls_order, bar_display, pang_tissues, "topk_bars_v", vertical=True)
make_pang_bar(df_pang_cls, "auprc", "AUPRC", pang_cls_order, bar_display, pang_tissues, "auprc_bars_v", vertical=True)

# F1
make_pang_bar(df_pang_cls, "f1", "F1", pang_cls_order, bar_display, pang_tissues, "f1_bars")
make_pang_bar(df_pang_cls, "f1", "F1", pang_cls_order, bar_display, pang_tissues, "f1_bars_v", vertical=True)

# --- regression bar plots (r2, pearson) ---
print("\n=== pangolin test regression ===")
for subset_name, subset_label in [("ssu_valid", "all sites"), ("ssu_valid_nonzero", "SSU > 0")]:
    sub_reg = df_pang_reg[df_pang_reg.subset == subset_name]
    suffix = "all" if subset_name == "ssu_valid" else "nonzero"

    make_pang_bar(sub_reg, "r2", r"$R^2$" + f" ({subset_label})",
                  pang_reg_order, r2_display, pang_tissues, f"r2_bars_{suffix}")
    make_pang_bar(sub_reg, "pearson", f"Pearson r ({subset_label})",
                  pang_reg_order, r2_display, pang_tissues, f"pearson_bars_{suffix}")

    # vertical
    make_pang_bar(sub_reg, "r2", r"$R^2$" + f" ({subset_label})",
                  pang_reg_order, r2_display, pang_tissues, f"r2_bars_{suffix}_v", vertical=True)
    make_pang_bar(sub_reg, "pearson", f"Pearson r ({subset_label})",
                  pang_reg_order, r2_display, pang_tissues, f"pearson_bars_{suffix}_v", vertical=True)

# --- individual per-tissue bar plots ---
print("\n=== individual per-tissue plots ===")
for tissue in pang_tissues:
    # classification
    for metric, ylabel in [("topk", "Top-k"), ("auprc", "AUPRC"), ("f1", "F1")]:
        tdf = df_pang_cls[(df_pang_cls.tissue == tissue) &
                          df_pang_cls.output.isin(pang_cls_order)]
        if tdf.empty:
            print(f"  {ylabel} {tissue}: no data, skipping")
            continue
        tdf = tdf.set_index("output").reindex(pang_cls_order)

        fig, ax = plt.subplots(1, 1, figsize=(3, 4))
        x = np.arange(len(pang_cls_order))
        colors = [get_color(o) for o in pang_cls_order]
        vals = tdf[metric].values
        bars = ax.bar(x, vals, color=colors, edgecolor="black", linewidth=0.5, alpha=0.8)
        for bar, v in zip(bars, vals):
            if np.isfinite(v):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                        f"{v:.2f}", ha="center", va="center", fontweight="bold",
                        rotation=90, color="white")
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        ax.set_title(pang_tissue_display.get(tissue, tissue))
        ax.set_xticks(x)
        ax.set_xticklabels([bar_display.get(o, o) for o in pang_cls_order],
                           rotation=45, ha="right")
        ax.set_ylim(0, 1.1)
        plt.tight_layout()
        plt.savefig(fig2_main / f"pang_{metric}_{tissue}.png", dpi=300, bbox_inches="tight")
        plt.show()

    # regression
    for subset_name, suffix in [("ssu_valid", "all"), ("ssu_valid_nonzero", "nonzero")]:
        sub_reg = df_pang_reg[(df_pang_reg.subset == subset_name) &
                              (df_pang_reg.tissue == tissue) &
                              df_pang_reg.output.isin(pang_reg_order)]
        if sub_reg.empty:
            continue
        sub_reg = sub_reg.set_index("output").reindex(pang_reg_order)

        for metric, ylabel in [("r2", r"$R^2$"), ("pearson", "Pearson r")]:
            fig, ax = plt.subplots(1, 1, figsize=(3, 4))
            x = np.arange(len(pang_reg_order))
            colors = [get_color(o) for o in pang_reg_order]
            vals = sub_reg[metric].values
            bars = ax.bar(x, vals, color=colors, edgecolor="black", linewidth=0.5, alpha=0.8)
            for bar, v in zip(bars, vals):
                if np.isfinite(v):
                    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                            f"{v:.2f}", ha="center", va="center", fontweight="bold",
                            rotation=90, color="white")
            ax.set_xlabel("")
            ax.set_ylabel(ylabel)
            ax.set_title(pang_tissue_display.get(tissue, tissue))
            ax.set_xticks(x)
            ax.set_xticklabels([r2_display.get(o, o) for o in pang_reg_order],
                               rotation=45, ha="right")
            plt.tight_layout()
            plt.savefig(fig2_main / f"pang_{metric}_{suffix}_{tissue}.png",
                        dpi=300, bbox_inches="tight")
            plt.show()

print(f"\nsaved pangolin test bar plots to {fig2_main}")

In [ ]:
# combined figure: AUPRC bars | binned AUPRC | binned MSE | R² swarm

# reorder tissues_fig1 to desired order via substring matching
_desired_order = ["haec", "lung", "brain", "testis", "blood"]
def _match_tissue(desired, available):
    """match desired short name to actual tissue name via substring"""
    for t in available:
        if desired in t.lower():
            return t
    return None
tissues_comb = [m for d in _desired_order if (m := _match_tissue(d, tissues_fig1)) is not None]
if not tissues_comb:
    print(f"WARNING: no tissue matches. tissues_fig1={tissues_fig1}")
    tissues_comb = tissues_fig1
print(f"tissues_fig1: {tissues_fig1}")
print(f"tissues_comb: {tissues_comb}")
print(f"tissues in df_cls: {sorted(df_cls.tissue.unique())}")
nt = len(tissues_comb)

# temporarily increase all text sizes for this large figure
_orig_params = {k: plt.rcParams[k] for k in ["font.size", "axes.titlesize", "axes.labelsize",
                "xtick.labelsize", "ytick.labelsize", "legend.fontsize", "legend.title_fontsize", "figure.titlesize"]}
_scale = 1.4
for k, v in _orig_params.items():
    plt.rcParams[k] = v * _scale

# --- configurable gaps between columns ---
_gap_01 = 0.35  # AUPRC bars ↔ Binned AUPRC
_gap_12 = 0.65  # Binned AUPRC ↔ Binned MSE
_gap_23 = 0.65  # Binned MSE ↔ R²

col_titles = ["AUPRC", "Binned AUPRC", "Binned MSE", r"$R^2$"]

# precompute bar data
_bar_df = df_cls[df_cls.subset == "all"].copy()
_bar_df = _bar_df.groupby(["tissue", "sample_id", "output"]).agg(val=("auprc", "mean")).reset_index()
_bar_df["base_model"] = _bar_df.output.apply(_get_base_model)
_bar_df = _bar_df[~_bar_df.base_model.isin(exclude_bases)]
_bar_avail = [o for o in bar_outputs if o in _bar_df.output.unique()]
_bar_df = _bar_df[_bar_df.output.isin(_bar_avail)]
_bar_order = [o for o in output_order_bar if o in _bar_avail]
_bar_order_compact = _bar_order

# precompute R² data
_r2_sub = df_reg[df_reg.subset == "ssu_valid"].copy()
_r2_sub["base_model"] = _r2_sub.output.apply(_get_base_model)
_r2_sub = _r2_sub[~_r2_sub.base_model.isin(exclude_bases)]
_r2_avail = [o for o in r2_outputs if o in _r2_sub.output.unique()]
_r2_sub = _r2_sub[_r2_sub.output.isin(_r2_avail)]
_r2_order = list(_r2_sub.groupby("output").r2.mean().sort_values(ascending=False).index)

_metric_col = "mae" if "mae" in df_bin_reg.columns else "mse"
_bin_labels = [f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)]

# precompute per-column y-axis limits (shared across rows)
_mse_vals = []
for t in tissues_comb:
    sub_m = df_bin_reg[(df_bin_reg.tissue == t) & (df_bin_reg.subset == "ssu_valid_nonzero")]
    if len(sub_m) > 0:
        avail_m = [o for o in output_order_spec if o in sub_m.output.unique().tolist()]
        sub_m = sub_m[sub_m.output.isin(avail_m)]
        agg_m = sub_m.groupby(["output", "bin"]).agg(metric=(_metric_col, "mean")).reset_index()
        _mse_vals.extend(agg_m["metric"].dropna().tolist())
_mse_ylim = (0, max(_mse_vals) * 1.1) if _mse_vals else (0, 1)

_r2_vals = []
for t in tissues_comb:
    tdf_r = _r2_sub[_r2_sub.tissue == t]
    if len(tdf_r) > 0:
        _r2_vals.extend(tdf_r.r2.tolist())
_r2_ylim = (min(_r2_vals) - 0.02, max(_r2_vals) + 0.02) if _r2_vals else (0, 1)
print(f"shared y-limits: MSE={_mse_ylim}, R²={_r2_ylim}")

_sec_fs = plt.rcParams["font.size"] * 0.75  # smaller font for secondary axis label

def _plot_row(fig, gs, row, tissue, show_titles=False, show_xlabels=False):
    """plot one tissue row into the given gridspec row"""
    _dc = [0, 2, 4, 6]

    # --- col 0: AUPRC bars ---
    ax = fig.add_subplot(gs[row, _dc[0]])
    tdf = _bar_df[_bar_df.tissue == tissue]
    agg = tdf.groupby("output").agg(mean=("val", "mean")).reset_index()
    agg = agg.set_index("output").reindex(_bar_order).reset_index()
    x = np.arange(len(_bar_order))
    bars = ax.bar(x, agg["mean"], color=[get_color(o) for o in _bar_order],
                  edgecolor="black", linewidth=0.5, alpha=0.8)
    for b, v in zip(bars, agg["mean"]):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() * 0.5,
                f"{v:.2f}", ha="center", va="center", fontweight="bold",
                rotation=90, color="white", fontsize=plt.rcParams["font.size"] * 1.25)
    ax.set_ylim(0, 1.1)
    ax.set_xticks(x)
    if show_xlabels:
        ax.set_xticklabels([bar_display.get(o, o) for o in _bar_order], rotation=45, ha="right")
    else:
        ax.set_xticklabels([])
    ax.set_ylabel("AUPRC")
    ax.text(-0.35, 0.5, tissue_display.get(tissue, tissue), transform=ax.transAxes,
            ha="right", va="center", fontweight="bold", fontsize=plt.rcParams["axes.labelsize"],
            rotation=90)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if show_titles:
        ax.set_title(col_titles[0], pad=15)

    # --- col 1: binned AUPRC ---
    ax = fig.add_subplot(gs[row, _dc[1]])
    sub = df_bin_ratio[(df_bin_ratio.tissue == tissue) & (df_bin_ratio.subset == "ssu_valid_nonzero")].copy()
    if len(sub) > 0:
        avail = [o for o in cls_order if o in sub.output.unique().tolist()]
        sub = sub[sub.output.isin(avail)]
        agg_b = sub.groupby(["output", "bin"]).agg(
            auprc=("auprc", "mean"), n_pos=("n_pos", "mean")).reset_index()
        counts = agg_b[agg_b.output == avail[0]][["bin", "n_pos"]].sort_values("bin")
        ax2 = ax.twinx()
        ax2.bar(counts["bin"], counts["n_pos"], width=0.5, color=GRAY_FAINT, alpha=0.4, zorder=1)
        ax2.tick_params(axis="y", labelcolor=GRAY_MED)
        ax2.set_yscale("log")
        ax2.set_ylim(counts["n_pos"].min() * 0.5, counts["n_pos"].max() * 3)
        ax2.set_ylabel("Splice Sites", color=GRAY_MED, fontsize=_sec_fs)
        for out in avail:
            out_df = agg_b[agg_b.output == out].sort_values("bin")
            ax.plot(out_df["bin"], out_df["auprc"], marker="o",
                    color=get_color(out), lw=2, markersize=4, zorder=3)
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)
    ax.set_ylim(0, 1)
    ax.set_ylabel("AUPRC")
    ax.set_xticks(range(10))
    if show_xlabels:
        ax.set_xticklabels(_bin_labels, rotation=45, ha="right")
        ax.set_xlabel("SSU bin")
    else:
        ax.set_xticklabels([])
    if show_titles:
        ax.set_title(col_titles[1], pad=15)

    # --- col 2: binned MSE ---
    ax = fig.add_subplot(gs[row, _dc[2]])
    sub = df_bin_reg[(df_bin_reg.tissue == tissue) & (df_bin_reg.subset == "ssu_valid_nonzero")].copy()
    if len(sub) > 0:
        avail = [o for o in output_order_spec if o in sub.output.unique().tolist()]
        sub = sub[sub.output.isin(avail)]
        agg_b = sub.groupby(["output", "bin"]).agg(
            metric=(_metric_col, "mean"), n=("n", "mean")).reset_index()
        counts = agg_b[agg_b.output == avail[0]][["bin", "n"]].sort_values("bin")
        ax2 = ax.twinx()
        ax2.bar(counts["bin"], counts["n"], width=0.5, color=GRAY_FAINT, alpha=0.4, zorder=1)
        ax2.tick_params(axis="y", labelcolor=GRAY_MED)
        ax2.set_yscale("log")
        ax2.set_ylim(counts["n"].min() * 0.5, counts["n"].max() * 3)
        ax2.set_ylabel("Splice Sites", color=GRAY_MED, fontsize=_sec_fs)
        for out in avail:
            out_df = agg_b[agg_b.output == out].sort_values("bin")
            is_reg = out in reg_outputs
            ax.plot(out_df["bin"], out_df["metric"], marker="o",
                    color=get_color(out), lw=2, markersize=4, zorder=3,
                    ls="--" if is_reg else "-")
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)
    ax.set_ylabel("MSE")
    ax.set_ylim(*_mse_ylim)
    ax.set_xticks(range(10))
    if show_xlabels:
        ax.set_xticklabels(_bin_labels, rotation=45, ha="right")
        ax.set_xlabel("SSU bin")
    else:
        ax.set_xticklabels([])
    if show_titles:
        ax.set_title(col_titles[2], pad=15)

    # --- col 3: R² swarm ---
    ax = fig.add_subplot(gs[row, _dc[3]])
    tdf = _r2_sub[_r2_sub.tissue == tissue]
    if len(tdf) > 0:
        sns.stripplot(data=tdf, x="output", y="r2", hue="output", order=_r2_order,
                      palette={o: get_color(o) for o in _r2_order},
                      size=4, jitter=0.2, alpha=0.7, ax=ax, legend=False)
        for i, out in enumerate(_r2_order):
            vals = tdf[tdf.output == out].r2
            if len(vals) == 0:
                continue
            ax.hlines(vals.mean(), i - 0.3, i + 0.3, color=get_color(out), lw=2)
            ax.hlines(vals.median(), i - 0.3, i + 0.3, color=get_color(out), lw=1.5, ls="--")
    ax.set_xlabel("")
    ax.set_ylabel(r"$R^2$")
    ax.set_ylim(*_r2_ylim)
    ax.set_xticks(range(len(_r2_order)))
    if show_xlabels:
        ax.set_xticklabels([r2_display.get(o, o) for o in _r2_order], rotation=45, ha="right")
    else:
        ax.set_xticklabels([])
    if show_titles:
        ax.set_title(col_titles[3], pad=15)

# ===== COMBINED FIGURE =====
fig = plt.figure(figsize=(28, 4.5 * nt))
gs = fig.add_gridspec(nt, 7,
                      width_ratios=[1, _gap_01, 1.5, _gap_12, 1.5, _gap_23, 1],
                      hspace=0.35, wspace=0.05)

for row, tissue in enumerate(tissues_comb):
    _plot_row(fig, gs, row, tissue,
              show_titles=(row == 0),
              show_xlabels=(row == nt - 1))

# restore original text sizes before saving
for k, v in _orig_params.items():
    plt.rcParams[k] = v

plt.savefig(fig2_main / "combined_vertical.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"saved {fig2_main}/combined_vertical.png")

# ===== INDIVIDUAL TISSUE FIGURES =====
for tissue in tissues_comb:
    # re-apply scaled text sizes
    for k, v in _orig_params.items():
        plt.rcParams[k] = v * _scale

    fig_t = plt.figure(figsize=(28, 5.5))
    gs_t = fig_t.add_gridspec(1, 7,
                               width_ratios=[1, _gap_01, 1.5, _gap_12, 1.5, _gap_23, 1],
                               wspace=0.05)
    _plot_row(fig_t, gs_t, 0, tissue, show_titles=True, show_xlabels=True)

    # restore text sizes
    for k, v in _orig_params.items():
        plt.rcParams[k] = v

    tname = tissue.lower().replace(" ", "_").replace("-", "_")
    fig_t.savefig(fig2_main / f"combined_{tname}.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"saved {fig2_main}/combined_{tname}.png")

# ===== STANDALONE LEGEND =====
from matplotlib.lines import Line2D

# build legend entries: group by base model, solid for cls, dashed for reg
# use output_order_spec which has all outputs shown in binned plots
_all_outputs = [o for o in output_order_spec
                if o in df_bin_reg[df_bin_reg.subset == "ssu_valid_nonzero"].output.unique().tolist()]
# group: collect (base_model, [(output, ls)]) preserving order
from collections import OrderedDict
_legend_groups = OrderedDict()
for o in _all_outputs:
    base = _get_base_model(o)
    if base not in _legend_groups:
        _legend_groups[base] = []
    ls = "--" if o in reg_outputs else "-"
    _legend_groups[base].append((o, ls))

# base model → display name
_base_display = {
    "sphaec_ref": "SPLAIRE", "pangolin": "Pangolin",
    "splicetransformer": "SpliceTransformer", "spliceai": "SpliceAI",
}

# build handles with spacers between groups
_leg_handles = []
_leg_labels = []
for i, (base, entries) in enumerate(_legend_groups.items()):
    if i > 0:
        # invisible spacer between groups
        _leg_handles.append(Line2D([0], [0], color="none", lw=0))
        _leg_labels.append("")
    for o, ls in entries:
        name = _base_display.get(base, base)
        display = f"{name} - SSU" if ls == "--" else name
        _leg_handles.append(Line2D([0], [0], color=get_color(o), lw=2.5,
                                   ls=ls, marker="o", markersize=8))
        _leg_labels.append(display)

fig_leg, ax_leg = plt.subplots(figsize=(3, 0.5 * len(_leg_labels)))
ax_leg.axis("off")
ax_leg.legend(handles=_leg_handles, labels=_leg_labels, loc="center",
              frameon=True, edgecolor="lightgray", fancybox=True,
              fontsize=plt.rcParams["font.size"] * _scale,
              handlelength=3, handleheight=1.5, labelspacing=0.6)
fig_leg.savefig(fig2_main / "combined_legend.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"saved {fig2_main}/combined_legend.png")

In [ ]:
# Option A: tissue-per-row, all 5 expression tissues
# columns: AUPRC bars | Binned AUPRC | R² swarm | Binned MSE (CLS) | Binned MSE (REG)

_all_compact_tissues = [t for t in tissues if t in df_bin_reg.tissue.unique()]
_mse_cls_heads = [o for o in output_order_spec if o not in reg_outputs]
_mse_reg_heads = [o for o in output_order_spec if o in reg_outputs]

n_tissues = len(_all_compact_tissues)
print(f"tissues: {_all_compact_tissues}")

for k, v in _orig_params.items():
    plt.rcParams[k] = v * _scale

fig, axes = plt.subplots(n_tissues, 5, figsize=(30, 5 * n_tissues))

for ti, tissue in enumerate(_all_compact_tissues):
    t_label = tissue_display.get(tissue, tissue)

    # --- col 0: AUPRC bars ---
    ax = axes[ti, 0]
    tdf = _bar_df[_bar_df.tissue == tissue]
    agg = tdf.groupby("output").agg(mean=("val", "mean")).reset_index()
    agg = agg.set_index("output").reindex(_bar_order_compact).reset_index()
    x = np.arange(len(_bar_order_compact))
    bars = ax.bar(x, agg["mean"], color=[get_color(o) for o in _bar_order_compact],
                  edgecolor="black", linewidth=0.5, alpha=0.8)
    for b, v in zip(bars, agg["mean"]):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() * 0.5,
                f"{v:.2f}", ha="center", va="center", fontweight="bold",
                rotation=90, color="white", fontsize=plt.rcParams["font.size"] * 1.25)
    ax.set_ylim(0, 1.1)
    ax.set_xticks(x)
    ax.set_xticklabels([bar_display.get(o, o) for o in _bar_order_compact], rotation=45, ha="right")
    ax.set_ylabel(f"{t_label}\nAUPRC")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ti == 0:
        ax.set_title("AUPRC", fontweight="bold")

    # --- col 1: binned AUPRC ---
    ax = axes[ti, 1]
    sub = df_bin_ratio[(df_bin_ratio.tissue == tissue) & (df_bin_ratio.subset == "ssu_valid_nonzero")].copy()
    if len(sub) > 0:
        avail = [o for o in cls_order if o in sub.output.unique().tolist()]
        sub = sub[sub.output.isin(avail)]
        agg_b = sub.groupby(["output", "bin"]).agg(
            auprc=("auprc", "mean"), n_pos=("n_pos", "mean")).reset_index()
        counts = agg_b[agg_b.output == avail[0]][["bin", "n_pos"]].sort_values("bin")
        ax2 = ax.twinx()
        ax2.bar(counts["bin"], counts["n_pos"], width=0.5, color=GRAY_FAINT, alpha=0.4, zorder=1)
        ax2.tick_params(axis="y", labelcolor=GRAY_MED)
        ax2.set_yscale("log")
        ax2.set_ylim(counts["n_pos"].min() * 0.5, counts["n_pos"].max() * 3)
        ax2.set_ylabel("Splice Sites", color=GRAY_MED, fontsize=_sec_fs)
        for out in avail:
            out_df = agg_b[agg_b.output == out].sort_values("bin")
            ax.plot(out_df["bin"], out_df["auprc"], marker="o",
                    color=get_color(out), lw=2, markersize=4, zorder=3)
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)
    ax.set_ylim(0, 1)
    ax.set_ylabel("AUPRC")
    ax.set_xticks(range(10))
    ax.set_xticklabels(_bin_labels, rotation=45, ha="right")
    ax.set_xlabel("SSU bin")
    if ti == 0:
        ax.set_title("Binned AUPRC", fontweight="bold")

    # --- col 2: R² swarm ---
    ax = axes[ti, 2]
    tdf = _r2_sub[_r2_sub.tissue == tissue]
    if len(tdf) > 0:
        sns.stripplot(data=tdf, x="output", y="r2", hue="output", order=_r2_order,
                      palette={o: get_color(o) for o in _r2_order},
                      size=5, jitter=0.2, alpha=0.7, ax=ax, legend=False)
        for i, out in enumerate(_r2_order):
            vals = tdf[tdf.output == out].r2
            if len(vals) == 0:
                continue
            ax.hlines(vals.mean(), i - 0.3, i + 0.3, color=get_color(out), lw=2)
            ax.hlines(vals.median(), i - 0.3, i + 0.3, color=get_color(out), lw=1.5, ls="--")
    ax.set_xlabel("")
    ax.set_ylabel(r"$R^2$")
    ax.set_ylim(*_r2_ylim)
    ax.set_xticks(range(len(_r2_order)))
    ax.set_xticklabels([r2_display.get(o, o) for o in _r2_order], rotation=45, ha="right")
    if ti == 0:
        ax.set_title(r"$R^2$", fontweight="bold")

    # --- col 3: binned MSE — CLS heads ---
    ax = axes[ti, 3]
    sub = df_bin_reg[(df_bin_reg.tissue == tissue) & (df_bin_reg.subset == "ssu_valid_nonzero")].copy()
    if len(sub) > 0:
        avail = [o for o in _mse_cls_heads if o in sub.output.unique().tolist()]
        sub_h = sub[sub.output.isin(avail)]
        if len(sub_h) > 0:
            agg_b = sub_h.groupby(["output", "bin"]).agg(
                metric=(_metric_col, "mean"), n=("n", "mean")).reset_index()
            for out in avail:
                out_df = agg_b[agg_b.output == out].sort_values("bin")
                ax.plot(out_df["bin"], out_df["metric"], marker="o",
                        color=get_color(out), lw=2, markersize=4, zorder=3)
    ax.set_ylabel("MSE")
    ax.set_ylim(*_mse_ylim)
    ax.set_xticks(range(10))
    ax.set_xticklabels(_bin_labels, rotation=45, ha="right")
    ax.set_xlabel("SSU bin")
    ax.grid(axis="y", alpha=0.2)
    if ti == 0:
        ax.set_title("Binned MSE (CLS)", fontweight="bold")

    # --- col 4: binned MSE — REG heads ---
    ax = axes[ti, 4]
    if len(sub) > 0:
        avail = [o for o in _mse_reg_heads if o in sub.output.unique().tolist()]
        sub_h = sub[sub.output.isin(avail)]
        if len(sub_h) > 0:
            agg_b = sub_h.groupby(["output", "bin"]).agg(
                metric=(_metric_col, "mean"), n=("n", "mean")).reset_index()
            for out in avail:
                out_df = agg_b[agg_b.output == out].sort_values("bin")
                ax.plot(out_df["bin"], out_df["metric"], marker="o",
                        color=get_color(out), lw=2, markersize=4, zorder=3,
                        ls="--")
    ax.set_ylabel("MSE")
    ax.set_ylim(*_mse_ylim)
    ax.set_xticks(range(10))
    ax.set_xticklabels(_bin_labels, rotation=45, ha="right")
    ax.set_xlabel("SSU bin")
    ax.grid(axis="y", alpha=0.2)
    if ti == 0:
        ax.set_title("Binned MSE (REG)", fontweight="bold")

for k, v in _orig_params.items():
    plt.rcParams[k] = v

plt.tight_layout()
plt.savefig(fig2_main / "compact_all_tissues.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"saved {fig2_main}/compact_all_tissues.png")


## Calibration Analysis

SSU calibration: predicted vs actual SSU across all model outputs.

In [ ]:
# SSU calibration: predicted vs actual SSU across all model outputs
# bins predictions, computes mean actual SSU per bin → perfect calibration = diagonal
import sys, pickle, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

sys.path.insert(0, "score/src")
from h5io import load_individual, add_sphaec_avg, find_model_files

cache_path = Path("dists/data/calibration_cache.pkl")
pred_base = Path("/scratch/runyan.m/sphaec_out")
n_cal_bins = 20
cal_edges = np.linspace(0, 1, n_cal_bins + 1)
SENTINEL = 777.0
N_WORKERS = 4

cal_tissues = [t for t in tissues_fig1 if t != "whole_blood"]

def _process_individual(pred_dir, cal_edges, n_cal_bins, SENTINEL):
    """load one individual and compute calibration bins — runs in subprocess"""
    import numpy as np
    from h5io import load_individual, add_sphaec_avg

    sid = pred_dir.name
    t0 = time.time()
    truth, preds = load_individual(pred_dir)
    add_sphaec_avg(preds)
    load_time = time.time() - t0

    is_acc = truth["acceptor"] == 1
    is_don = truth["donor"] == 1
    ssu = truth["ssu"]
    valid = (ssu != SENTINEL) & (ssu > 0)

    site_idx = np.flatnonzero((is_acc | is_don) & valid)
    acc_idx = np.flatnonzero(is_acc & valid)
    don_idx = np.flatnonzero(is_don & valid)
    y_true_all = ssu[site_idx].astype(np.float32)

    sample_cal = {}

    for model, p in preds.items():
        # CLS output: matched acc/don predictions
        if "acceptor" in p and "donor" in p:
            col = f"{model}_cls"
            y_pred = np.concatenate([p["acceptor"][acc_idx], p["donor"][don_idx]]).astype(np.float32)
            y_true_cls = np.concatenate([ssu[acc_idx], ssu[don_idx]]).astype(np.float32)

            bin_idx = np.clip(np.digitize(y_pred, cal_edges) - 1, 0, n_cal_bins - 1)
            pm, tm, ct = [], [], []
            for b in range(n_cal_bins):
                m = bin_idx == b
                n = m.sum()
                ct.append(int(n))
                pm.append(float(y_pred[m].mean()) if n > 0 else np.nan)
                tm.append(float(y_true_cls[m].mean()) if n > 0 else np.nan)
            sample_cal[col] = {"pred_mean": pm, "true_mean": tm, "count": ct}

        # single-score outputs
        for key, arr in p.items():
            if key in ["acceptor", "donor", "neither", "splice"]:
                continue
            col = f"{model}_{key}"
            y_pred = arr[site_idx].astype(np.float32)

            bin_idx = np.clip(np.digitize(y_pred, cal_edges) - 1, 0, n_cal_bins - 1)
            pm, tm, ct = [], [], []
            for b in range(n_cal_bins):
                m = bin_idx == b
                n = m.sum()
                ct.append(int(n))
                pm.append(float(y_pred[m].mean()) if n > 0 else np.nan)
                tm.append(float(y_true_all[m].mean()) if n > 0 else np.nan)
            sample_cal[col] = {"pred_mean": pm, "true_mean": tm, "count": ct}

    elapsed = time.time() - t0
    return sid, sample_cal, len(site_idx), len(sample_cal), elapsed, load_time

if cache_path.exists():
    with open(cache_path, "rb") as f:
        cal_data = pickle.load(f)
    print(f"loaded cached calibration from {cache_path}")
else:
    cal_data = {}

needs_compute = [t for t in cal_tissues if t not in cal_data]
if needs_compute:
    print(f"need to compute: {needs_compute}")
    print(f"using {N_WORKERS} parallel workers")
    t0_all = time.time()

    for ti, tissue in enumerate(needs_compute):
        t0_tissue = time.time()

        # discover prediction directories
        pred_root = None
        for pattern in ["ml_out/predictions", "ml_out"]:
            candidate = pred_base / tissue / pattern
            if candidate.exists():
                test_dirs = [d for d in sorted(candidate.iterdir()) if d.is_dir() and find_model_files(d)]
                if test_dirs:
                    pred_root = candidate
                    break
        if pred_root is None:
            print(f"{tissue}: no prediction directories found under {pred_base / tissue}")
            continue

        pred_dirs = sorted(d for d in pred_root.iterdir() if d.is_dir() and find_model_files(d))
        n_dirs = len(pred_dirs)
        print(f"\n{'='*60}")
        print(f"[{ti+1}/{len(needs_compute)}] {tissue}: {n_dirs} individuals in {pred_root}")
        print(f"{'='*60}")

        tissue_cal = {}
        done = 0
        with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
            futures = {pool.submit(_process_individual, d, cal_edges, n_cal_bins, SENTINEL): d
                       for d in pred_dirs}
            for fut in as_completed(futures):
                done += 1
                try:
                    sid, sample_cal, n_sites, n_outputs, elapsed, load_time = fut.result()
                    tissue_cal[sid] = sample_cal
                    elapsed_tissue = time.time() - t0_tissue
                    remaining = (elapsed_tissue / done) * (n_dirs - done)
                    print(f"  [{done}/{n_dirs}] {sid}: {n_sites:,} sites, {n_outputs} outputs, "
                          f"{elapsed:.1f}s (load {load_time:.1f}s) | tissue ETA {remaining/60:.1f}min")
                except Exception as e:
                    sid = futures[fut].name
                    print(f"  [{done}/{n_dirs}] {sid}: FAILED ({e})")

        cal_data[tissue] = tissue_cal
        tissue_time = time.time() - t0_tissue
        print(f"{tissue} done: {len(tissue_cal)} individuals in {tissue_time/60:.1f}min")

        # checkpoint after each tissue
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cache_path, "wb") as f:
            pickle.dump(cal_data, f)
        print(f"  checkpoint saved to {cache_path}")

    total_time = time.time() - t0_all
    print(f"\ntotal: {total_time/60:.1f}min")
else:
    print("all tissues cached")

# summary
print(f"\n{'='*60}")
for tissue in cal_data:
    n_ind = len(cal_data[tissue])
    outputs = set()
    for sid in cal_data[tissue]:
        outputs.update(cal_data[tissue][sid].keys())
    print(f"{tissue}: {n_ind} individuals, {len(outputs)} outputs")

In [ ]:
# calibration plots: predicted SSU vs actual SSU
# separate panels for CLS outputs and SSU/usage outputs

cal_cls_outputs = [
    "spliceai_cls",
    "sphaec_var_cls",
    "pangolin_avg_p_splice",
    "splicetransformer_cls",
]
cal_ssu_outputs = [
    "sphaec_var_ssu",
    "pangolin_avg_usage",
    "splicetransformer_avg_tissue",
]

cal_display = {
    "sphaec_var_ssu": "SPLAIRE SSU",
    "sphaec_var_cls": "SPLAIRE CLS",
    "pangolin_avg_usage": "Pangolin Avg Usage",
    "pangolin_avg_p_splice": "Pangolin Avg CLS",
    "splicetransformer_avg_tissue": "SpliceTransformer Avg Tissue",
    "splicetransformer_cls": "SpliceTransformer CLS",
    "spliceai_cls": "SpliceAI CLS",
}

def _cal_curve(tdata, output):
    """weighted-mean calibration curve across individuals"""
    all_pred, all_true, all_count = [], [], []
    for sid, scal in tdata.items():
        if output not in scal:
            continue
        all_pred.append(scal[output]["pred_mean"])
        all_true.append(scal[output]["true_mean"])
        all_count.append(scal[output]["count"])
    if not all_pred:
        return None, None
    pred_arr = np.array(all_pred)
    true_arr = np.array(all_true)
    count_arr = np.array(all_count, dtype=float)
    with np.errstate(invalid="ignore"):
        count_arr[np.isnan(pred_arr)] = 0
        w_sum = count_arr.sum(axis=0)
        pred_mean = np.where(w_sum > 0, np.nansum(pred_arr * count_arr, axis=0) / w_sum, np.nan)
        true_mean = np.where(w_sum > 0, np.nansum(true_arr * count_arr, axis=0) / w_sum, np.nan)
    valid = np.isfinite(pred_mean) & np.isfinite(true_mean)
    return pred_mean[valid], true_mean[valid]

nt = len(cal_tissues)
pw = max(7, 6 * nt)

# --- CLS calibration ---
fig, axes = plt.subplots(1, nt, figsize=(pw, 7), sharey=True)
if nt == 1:
    axes = [axes]

for ti, tissue in enumerate(cal_tissues):
    ax = axes[ti]
    tdata = cal_data.get(tissue, {})

    for output in cal_cls_outputs:
        pm, tm = _cal_curve(tdata, output)
        if pm is None:
            continue
        ax.plot(pm, tm, marker="o", markersize=6, color=get_color(output),
                lw=2.5, label=cal_display.get(output, output))

    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_xlabel("Predicted Score", fontsize=14)
    if ti == 0:
        ax.set_ylabel("Actual SSU", fontsize=14)
    ax.set_title(tissue_display.get(tissue, tissue), fontsize=15, fontweight="bold")
    ax.tick_params(labelsize=12)
    ax.grid(alpha=0.15)
    ax.legend(fontsize=12, loc="upper left", framealpha=0.9)

fig.suptitle("Calibration — Classification Outputs", fontsize=17, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(fig2_main / "calibration_cls.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"saved {fig2_main}/calibration_cls.png")

# --- SSU/usage calibration ---
fig, axes = plt.subplots(1, nt, figsize=(pw, 7), sharey=True)
if nt == 1:
    axes = [axes]

for ti, tissue in enumerate(cal_tissues):
    ax = axes[ti]
    tdata = cal_data.get(tissue, {})

    for output in cal_ssu_outputs:
        pm, tm = _cal_curve(tdata, output)
        if pm is None:
            continue
        ax.plot(pm, tm, marker="o", markersize=6, color=get_color(output),
                lw=2.5, ls="--", label=cal_display.get(output, output))

    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_xlabel("Predicted SSU", fontsize=14)
    if ti == 0:
        ax.set_ylabel("Actual SSU", fontsize=14)
    ax.set_title(tissue_display.get(tissue, tissue), fontsize=15, fontweight="bold")
    ax.tick_params(labelsize=12)
    ax.grid(alpha=0.15)
    ax.legend(fontsize=12, loc="upper left", framealpha=0.9)

fig.suptitle("Calibration — SSU/Usage Outputs", fontsize=17, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(fig2_main / "calibration_ssu.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"saved {fig2_main}/calibration_ssu.png")

## Figure 2: Model Performance Summary

**Caption:** Combined performance summary across evaluation metrics. (a) R² swarm plots showing per-tissue regression performance for predicting splice site usage. (b) Top-k accuracy bar plot for classification. (c) Binned AUPRC by true SSU level showing classification performance across the SSU spectrum. (d) Binned MAE showing regression accuracy across SSU levels.

**Key Points:**
- SPLAIRE achieves highest R² for SSU prediction, particularly for constitutive (high SSU) and alternative (mid SSU) splice sites
- Classification metrics (AUPRC, top-k) confirm strong performance in identifying functional splice sites
- Performance varies across SSU bins, with models generally performing better on high-usage sites
- Regression outputs from SPLAIRE better capture quantitative usage levels compared to classification-only models